In [2]:
%%capture
# LLM chay TU XA tren Modal GPU => notebook chi can client + retrieval (khong cai vLLM/torch nang o day).
# Chay 1 lan, KHONG can restart kernel -> Run All chay mot mach.
%uv pip install -q modal sentence-transformers 'vllm==0.25.1' 'transformers==5.14.1' python-dotenv pandas scikit-learn tqdm numpy


In [3]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Chay local + offload LLM sang Modal GPU. Can: MODAL_TOKEN_ID/MODAL_TOKEN_SECRET (Modal CLI hoac .env),
# HF_TOKEN (tai model o remote). Retrieval KHONG dung Qdrant nua: doc dieu luat tu file da chuan bi san.
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path('/root/.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/HF_TOKEN tu environment.')
# Tren Modal Notebooks: KHONG can MODAL_TOKEN_* (kernel da xac thuc san),
# va HF_TOKEN do Modal Secret bom vao environment.
for _k in ['HF_TOKEN']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'MISSING -> gan Modal Secret hoac dat bien moi truong')
for _k in ['MODAL_TOKEN_ID', 'MODAL_TOKEN_SECRET']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'trong (BINH THUONG neu chay ngay tren Modal Notebooks)')

# Ep ro MODAL_ENVIRONMENT -- sua loi NotFoundError('Environment not found') khi app.run()
# tu suy Environment tu context that bai (gap tren mot so tai khoan Modal Notebooks).
# Doi 'main' sang ten Environment khac (vd 'bi_mat_khong_the_bat_mi') neu can, truoc khi
# chay cell nay -- setdefault nen KHONG ghi de neu ban da tu set truoc do.
os.environ.setdefault('MODAL_ENVIRONMENT', 'main')
print('  MODAL_ENVIRONMENT:', os.environ['MODAL_ENVIRONMENT'])

# Goc project (noi co .env) -> dung de tim dataset & .env cho Modal, khong phu thuoc cwd cua kernel.
PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd()
print('PROJECT_ROOT:', PROJECT_ROOT)

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
AVAILABLE_MODELS = ['Qwen3.5-9B']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen3.5-9B': 'Qwen/Qwen3.5-9B',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'Qwen3.5-9B': {
        'profile_name': 'vendor_qwen3_5_thinking_general',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 1.0, 'top_p': 0.95,
            'top_k': 20, 'min_p': 0.0, 'repetition_penalty': 1.0,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
# Revision theo từng model. Base tự lấy từ model đã load; Modal phải chỉ định rõ,
# nên đổi model mà quên đổi revision sẽ gây 404 RevisionNotFound.
MODEL_REVISIONS = {
    'Qwen3-4B': '1cfa9a7208912126459214e8b04321603b3df60c',
    'Qwen2.5-7B-Instruct': 'main',
    'Llama-3.1-8B-Instruct': 'main',
    'Llama-3.2-3B': 'main',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'main',
    'DeepSeek-R1-Distill-Llama-8B': 'main',
    'Qwen3.5-9B': 'main',
}
MODEL_REVISION = os.getenv('MODEL_REVISION', MODEL_REVISIONS.get(MODEL_NAME, 'main'))
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v12_legal_pado_divisible_types_balanced_judge'
BENCHMARK_PROTOCOL = 'legal_pado_thinking_divisible_mapper_3vote_derived_verdict'
# ---- Retrieval: dung file dieu luat da chuan bi san, KHONG search Qdrant ----
# Doi nguon retrieval => doi retrieval_signature => doi pipeline_contract_hash va cache_key,
# nen ket qua cu tu dong bi tinh lai chu khong bi dung nham.
RETRIEVAL_FILE_NAME = os.getenv('ALQAC_RETRIEVAL_FILE_NAME', 'retrieval_top14_ours.json')
RETRIEVAL_SOURCE = f'file:{RETRIEVAL_FILE_NAME}'
TOP_K = int(os.getenv('ALQAC_TOP_K', '14'))
MAX_INPUT_TOKENS = 24000
PADO_PIPELINE_VERSION = 'legal-pado-v15-derived-verdict-span-recall'
# Thinking mode (khớp base) cần token cho reasoning + JSON; ceiling theo base = 8192.
MAX_NEW_TOKENS_ISSUE = 4096
MAX_NEW_TOKENS_JUDGE = 16384
MAX_NEW_TOKENS_DIRECT = 2048
JUDGE_VOTES = 3
MAX_NEW_TOKENS = max(MAX_NEW_TOKENS_ISSUE, MAX_NEW_TOKENS_JUDGE, MAX_NEW_TOKENS_DIRECT)

# ---- vLLM engine knobs ----
ENABLE_THINKING = ACTIVE_PROFILE['enable_thinking']  # theo profile của model (đổi model là tự khớp)
USE_STRUCTURED_OUTPUTS = False  # Base sinh JSON tự do + parse robust; grammar sẽ đổi phân phối decoding.
VLLM_DTYPE = os.getenv('VLLM_DTYPE', 'bfloat16')  # khớp base; dùng L4/A10/A100
assert VLLM_DTYPE == 'bfloat16', 'Base đã chạy BF16; đổi dtype sẽ làm mất tính so sánh.'
GPU_MEM_UTIL = float(os.getenv('VLLM_GPU_MEM_UTIL', '0.92'))
# Top-14 dai hon top-10 ~2.6x: prompt judge worst-case ~16.3k token, cong max_new 16384 thi
# vuot 32768. Qwen3 co max_position_embeddings=40960 nen nang tran thay vi cat noi dung luat.
# Neu engine tu choi 40960: dat VLLM_MAX_MODEL_LEN=32768 va ALQAC_MAX_ARTICLE_CHARS=4000.
MAX_MODEL_LEN = int(os.getenv('VLLM_MAX_MODEL_LEN', '40960'))
MAX_NUM_SEQS = int(os.getenv('VLLM_MAX_NUM_SEQS', '64'))
VLLM_ENFORCE_EAGER = os.getenv('VLLM_ENFORCE_EAGER', '0') == '1'
VLLM_RUNTIME_VERSION = '0.25.1'
# Qwen3 thinking vendor profile (khớp base); seed riêng cho từng vote để reproducible.
SAMPLING = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items() if k != 'do_sample'}
MAX_ARTICLE_CHARS = int(os.getenv('ALQAC_MAX_ARTICLE_CHARS', '6000'))  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # strict: mỗi stage chỉ được gọi đúng một lần
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    if required:
        raise RuntimeError(
            f'Thiếu secret {names}. Hãy gắn Modal Secret vào notebook hoặc khai báo biến môi trường tương ứng.'
        )
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)

# LLM chay tren Modal GPU (remote). Notebook chi can CPU: khong con buoc embedding local.
torch.manual_seed(EVAL_SEEDS[0])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Retrieval:', RETRIEVAL_SOURCE, '| top_k:', TOP_K, '| local torch:', torch.__version__)
print('Model (remote vLLM):', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('vLLM dtype:', VLLM_DTYPE, '| unquantized')


Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/HF_TOKEN tu environment.
  HF_TOKEN OK
  MODAL_TOKEN_ID OK
  MODAL_TOKEN_SECRET OK
  MODAL_ENVIRONMENT: main
PROJECT_ROOT: /root
Retrieval: file:retrieval_top14_ours.json | top_k: 14 | local torch: 2.11.0+cu130
Model (remote vLLM): Qwen3.5-9B -> Qwen/Qwen3.5-9B
Benchmark: v12_legal_pado_divisible_types_balanced_judge | protocol: legal_pado_thinking_divisible_mapper_3vote_derived_verdict
Generation profile: vendor_qwen3_5_thinking_general | seeds: [2026]
vLLM dtype: bfloat16 | unquantized


In [4]:
def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PUBLIC_TEST_PATH=/path/to/file.json
    candidates = []
    if os.getenv('ALQAC_PUBLIC_TEST_PATH'):
        candidates.append(Path(os.environ['ALQAC_PUBLIC_TEST_PATH']))
    candidates += [
        PROJECT_ROOT / 'data' / 'ALQAC2026_public_test.json',
        PROJECT_ROOT / 'ALQAC2026_public_test.json',
        Path.cwd() / 'ALQAC2026_public_test.json',
        Path('/root/ALQAC2026_public_test.json'),
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh/ALQAC2026_public_test.json'),
        Path('data/ALQAC2026_public_test.json'),
        Path('../data/ALQAC2026_public_test.json'),
        Path('/kaggle/working/ALQAC2026_public_test.json'),
    ]
    # --- Modal Notebooks: file ban upload thuong nam canh notebook (cwd) hoac trong home/root. ---
    candidates += [
        Path.home() / 'ALQAC2026_public_test.json',
        Path('/root/data/ALQAC2026_public_test.json'),
        Path('/workspace/ALQAC2026_public_test.json'),
        Path('/notebooks/ALQAC2026_public_test.json'),
    ]
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob('ALQAC2026_public_test.json'))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('ALQAC2026_public_test.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Không tìm thấy ALQAC2026_public_test.json. Đã kiểm tra:\n{checked}')


DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

assert len(public_data) == 50, f'Expected 50 cases, got {len(public_data)}'
assert len({x['case_id'] for x in public_data}) == len(public_data)
assert all(x.get('case_query') for x in public_data)
assert all(x.get('verdict_label') in LABELS for x in public_data)

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
# ---- Nap evidence bo sung (agent_v4_results.json); khai bao giong public test ----
def find_agent_evidence():
    candidates = []
    if os.getenv('ALQAC_AGENT_EVIDENCE_PATH'):
        candidates.append(Path(os.environ['ALQAC_AGENT_EVIDENCE_PATH']))
    for base in [PROJECT_ROOT / 'data', PROJECT_ROOT, Path.cwd(), Path.home(),
                 Path('/root'), Path('/root/data'), Path('/workspace'), Path('/notebooks')]:
        candidates.append(base / 'agent_v4_results.json')
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob('agent_v4_results.json'))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('agent_v4_results.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError('Khong tim thay agent_v4_results.json (evidence bo sung).')

AGENT_EVIDENCE_PATH = find_agent_evidence()
with AGENT_EVIDENCE_PATH.open(encoding='utf-8') as f:
    agent_evidence_raw = json.load(f)

# Gioi han so ky tu evidence dua vao prompt (tranh vuot MAX_INPUT_TOKENS). Chinh qua env neu can.
MAX_EVIDENCE_CHARS = int(os.getenv('ALQAC_MAX_EVIDENCE_CHARS', '8000'))

def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
    seen, parts, total = set(), [], 0
    for e in record.get('evidence_details', []):
        chunk_id = e.get('chunk_id')
        txt = str(e.get('text', '')).strip()
        if not txt or chunk_id in seen:
            continue
        seen.add(chunk_id)
        if total + len(txt) + 1 > max_chars:
            break
        parts.append(txt)
        total += len(txt) + 1
    return (chr(10)).join(parts)

evidence_by_case = {r['case_id']: _case_evidence_text(r) for r in agent_evidence_raw}


def _case_evidence_ids(record):
    """ID cac segment evidence cua case, GIU NGUYEN thu tu, bo trung.

    Them de KHOP cau truc voi ban PRIVATE (private/alqac-e2e-bge-m3-qwen35-9b-pado.ipynb,
    2026-07-24): submission yeu cau 'case_evidence' la danh sach id segment (vd
    'case_4101_seg_...'). Uu tien khoa 'evidence' cua agent_v4; neu thieu thi lay chunk_id
    trong 'evidence_details'.
    """
    ids, seen = [], set()
    for value in (record.get('evidence') or []):
        sid = str(value).strip()
        if sid and sid not in seen:
            seen.add(sid)
            ids.append(sid)
    if not ids:
        for detail in (record.get('evidence_details') or []):
            sid = str(detail.get('chunk_id', '')).strip()
            if sid and sid not in seen:
                seen.add(sid)
                ids.append(sid)
    return ids


evidence_ids_by_case = {r['case_id']: _case_evidence_ids(r) for r in agent_evidence_raw}
_missing_ev = [x['case_id'] for x in public_data if not evidence_by_case.get(x['case_id'])]
if _missing_ev:
    print('CANH BAO: thieu evidence cho case:', _missing_ev)
print('Evidence bo sung:', AGENT_EVIDENCE_PATH, '| phu', len(public_data) - len(_missing_ev), '/ 50 case')

inference_cases = [
    {
        'case_id': x['case_id'],
        'case_query': x['case_query'],
        'case_facts': evidence_by_case.get(x['case_id'], ''),
    }
    for x in public_data
]
gold_by_case = {x['case_id']: x['verdict_label'] for x in public_data}

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))
print('Retrieval: nap tu file o cell sau, khong ket noi Qdrant va khong load BGE-M3.')


Evidence bo sung: /root/agent_v4_results.json | phu 50 / 50 case
Dataset: /root/ALQAC2026_public_test.json
Cases: 50
Retrieval: nap tu file o cell sau, khong ket noi Qdrant va khong load BGE-M3.


In [5]:
# ==== Retrieval: NAP TU FILE, khong search Qdrant ====
# Truoc day cell nay encode case_query bang BGE-M3 roi query Qdrant lay top-10.
# Gio dieu luat cho tung case da duoc chuan bi san ngoai notebook. Cell chi doc, kiem tra
# va do vao retrieval_cache theo DUNG schema cu (rank/score/law_id/aid/article_no/content_Article)
# nen toan bo cac cell phia sau khong phai doi gi.
def find_retrieval_file():
    candidates = []
    if os.getenv('ALQAC_RETRIEVAL_PATH'):
        candidates.append(Path(os.environ['ALQAC_RETRIEVAL_PATH']))
    for base in [PROJECT_ROOT / 'data', PROJECT_ROOT, Path.cwd(), Path.cwd() / 'data',
                 Path.home(), Path('/root'), Path('/root/data'), Path('/workspace'), Path('/notebooks')]:
        candidates.append(base / RETRIEVAL_FILE_NAME)
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob(RETRIEVAL_FILE_NAME))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(RETRIEVAL_FILE_NAME))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(
        f'Khong tim thay {RETRIEVAL_FILE_NAME}. Dat file canh notebook, vao thu muc data/, '
        f'hoac tro duong dan bang ALQAC_RETRIEVAL_PATH.'
    )


LAW_FIELDS = ('rank', 'score', 'law_id', 'aid', 'article_no', 'content_Article')


def normalize_laws(case_id, items):
    """Giu nguyen thu tu trong file, bo trung (law_id, aid), danh lai rank 1..N."""
    if not isinstance(items, list):
        raise ValueError(f'{case_id}: gia tri phai la list dieu luat, nhan duoc {type(items).__name__}')
    laws, seen = [], set()
    for item in items:
        missing = [f for f in LAW_FIELDS if f not in item]
        if missing:
            raise ValueError(f'{case_id}: mot dieu luat thieu truong {missing}')
        key = (str(item['law_id']), int(item['aid']))
        if key in seen:
            continue
        seen.add(key)
        laws.append({
            'rank': len(laws) + 1,
            'score': float(item['score']),
            'law_id': key[0],
            'aid': key[1],
            'article_no': int(item['article_no']),
            'content_Article': str(item['content_Article'] or ''),
        })
    if not laws:
        raise ValueError(f'{case_id}: khong co dieu luat nao')
    return laws


RETRIEVAL_PATH = find_retrieval_file()
_raw_retrieval = json.loads(RETRIEVAL_PATH.read_text(encoding='utf-8'))
assert isinstance(_raw_retrieval, dict), 'File retrieval phai la dict {case_id: [dieu luat, ...]}'

expected_case_ids = {x['case_id'] for x in inference_cases}
_missing_cases = sorted(expected_case_ids - set(_raw_retrieval))
assert not _missing_cases, f'File retrieval thieu {len(_missing_cases)} case: {_missing_cases[:5]}'

retrieval_cache = {cid: normalize_laws(cid, _raw_retrieval[cid]) for cid in expected_case_ids}
_sizes = {len(v) for v in retrieval_cache.values()}
assert _sizes == {TOP_K}, f'So dieu luat/case = {sorted(_sizes)}, khong khop TOP_K={TOP_K}'
assert all(law['content_Article'].strip()
           for laws in retrieval_cache.values() for law in laws), 'Co dieu luat rong content_Article'

# Chu ky nay di vao manifest + pipeline contract + cache_key: doi file la moi case tu dong chay lai.
retrieval_signature = hashlib.sha256(
    json.dumps(
        {'source': RETRIEVAL_SOURCE, 'top_k': TOP_K, 'laws': retrieval_cache},
        ensure_ascii=False, sort_keys=True,
    ).encode('utf-8')
).hexdigest()

sample_case = inference_cases[0]
sample_laws = retrieval_cache[sample_case['case_id']]
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])

_law_chars = [sum(len(law['content_Article'][:MAX_ARTICLE_CHARS]) for law in laws)
              for laws in retrieval_cache.values()]
print('Retrieval file:', RETRIEVAL_PATH)
print('Cases:', len(retrieval_cache), '| dieu luat/case:', TOP_K,
      '| signature:', retrieval_signature[:12])
print('Law context sau khi cat', MAX_ARTICLE_CHARS, 'ky tu/dieu: TB',
      round(sum(_law_chars) / len(_law_chars)), '| MAX', max(_law_chars), 'ky tu')


,rank,score,law_id,article_no,aid
0,1,0.032787,91/2015/QH13,603,53373
1,2,0.031258,91/2015/QH13,584,53354
2,3,0.030622,91/2015/QH13,585,53355
3,4,0.028577,91/2015/QH13,599,53369
4,5,0.027778,91/2015/QH13,449,53219
5,6,0.027013,91/2015/QH13,419,53189
6,7,0.021917,92/2015/QH13,26,50691
7,8,0.500000,92/2015/QH13,147,50812
8,9,0.500000,92/2015/QH13,35,50700
9,10,0.500000,92/2015/QH13,39,50704


Retrieval file: /root/retrieval_top14_ours.json
Cases: 50 | dieu luat/case: 14 | signature: 92021d6a574e
Law context sau khi cat 6000 ky tu/dieu: TB 24290 | MAX 33396 ky tu


In [6]:
# ==== Robust compact structured vLLM engine — LOAD TRUC TIEP tren GPU cua kernel ====
# (2026-07-24) DOI KIEN TRUC: ban goc dung modal.App(...).run() de spin 1 container GPU
# RIENG (remote). Tren tai khoan Modal cua nhom, app.run() lien tuc bao
# NotFoundError('Environment not found') (xac nhan qua ca 1 doan test toi gian, khong
# lien quan gi code pado) -- kha nang do cau hinh workspace/token, KHONG sua duoc tu
# code. Trong khi kernel Notebook nay DA CO SAN GPU dung duoc that (xac nhan qua 1
# notebook khac load model truc tiep tren GPU kernel chay thanh cong). => bo han
# modal.App/RemoteLLM/app.run(), load thang vLLM engine trong CHINH tien trinh kernel.
# Moi logic sampling/schema/usage GIU NGUYEN 100% (chi doi tu goi .remote() sang goi
# thang local, dong bo, cung tien trinh) -- KHONG doi hanh vi sinh, chi doi noi chay.
import unicodedata

prompt_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, revision=MODEL_REVISION, trust_remote_code=True
)

MODEL_DTYPE = VLLM_DTYPE
# flashinfer JIT-compile sampler co the FAIL neu moi truong thieu CUDA dev headers ->
# ep sampler PyTorch thuan (giong config anh Modal cu tung dat qua .env() cho image).
os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '0')

GPU_TYPE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (KHONG CO GPU!)'
print(f'Loading {MODEL_ID} truc tiep tren GPU kernel ({GPU_TYPE}), khong qua Modal remote container...')
from vllm import LLM as _VllmLLM
vllm_engine = _VllmLLM(
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    revision=MODEL_REVISION,
    tokenizer_revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    trust_remote_code=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
    enforce_eager=VLLM_ENFORCE_EAGER,
    seed=EVAL_SEEDS[0],
)
print('vLLM ready (local, cung tien trinh voi notebook).')


def vllm_generate(messages_list, seeds, schema, max_tokens, enable_thinking=None):
    if len(messages_list) != len(seeds):
        raise ValueError('messages_list and seeds length mismatch')
    from vllm import SamplingParams
    try:
        from vllm.sampling_params import StructuredOutputsParams
    except ImportError:
        from vllm import StructuredOutputsParams

    structured = None
    if USE_STRUCTURED_OUTPUTS and schema is not None:
        structured = StructuredOutputsParams(
            json=schema,
            disable_any_whitespace=True,
        )

    def make_params(seed):
        kwargs = dict(max_tokens=max_tokens, seed=int(seed))
        for _k in ('temperature', 'top_p', 'top_k', 'min_p', 'repetition_penalty'):
            if _k in SAMPLING:
                kwargs[_k] = SAMPLING[_k]
        if structured is not None:
            kwargs['structured_outputs'] = structured
        return SamplingParams(**kwargs)

    tmpl_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        tmpl_kwargs['enable_thinking'] = ENABLE_THINKING if enable_thinking is None else enable_thinking
    outputs = vllm_engine.chat(
        messages_list,
        [make_params(seed) for seed in seeds],
        add_generation_prompt=True,
        chat_template_kwargs=tmpl_kwargs,
        use_tqdm=True,
    )
    results = []
    for output in outputs:
        generated = output.outputs[0]
        results.append((generated.text, {
            'input_tokens': len(output.prompt_token_ids),
            'output_tokens': len(generated.token_ids),
            'total_tokens': len(output.prompt_token_ids) + len(generated.token_ids),
            'hit_max_new_tokens': generated.finish_reason == 'length',
            'finish_reason': generated.finish_reason,
        }))
    return results


def close_modal():
    """Giai phong VRAM local (giu nguyen ten ham -- cell dong o cuoi notebook dang goi
    close_modal(), khong doi ten de khong phai sua cell khac)."""
    global vllm_engine
    try:
        del vllm_engine
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Da giai phong GPU local.')

def consensus_claim_outcomes(votes, issue_map):
    """Tang 1 (majority-at-claim): gop outcome cua TUNG claim qua cac phieu roi moi
    derive, thay vi derive tren mot phieu dai dien duy nhat.

    Ly do: tren run Qwen3.5-9B v15 chi 53% claim duoc ca 3 phieu cho cung outcome; chon
    1 phieu dai dien khuech dai nhieu sampling, con gop o muc claim dan nhieu o don vi
    nho nhat. Do doi chung (cung outcomes): phieu dai dien strict .580/F1 .547 ->
    majority-at-claim .640/.622 (paired bootstrap P(tot hon) = .939). Thuan hau ky.

    Tie-break khi cac phieu bat dong hoan toan: trong cac outcome thang phieu, chon
    outcome ON HOA nhat (a_share gan 0.65) - xac dinh, khong phu thuoc thu tu phieu.
    """
    a_share = {'FULL_ACCEPT': 1.0, 'PARTIAL_A_LEAN': 0.65, 'PARTIAL_B_LEAN': 0.35, 'REJECT': 0.0}
    order, tally = [], {}
    for vote in votes:
        for item in vote.get('claim_outcomes', []) or []:
            claim_id = item.get('claim_id')
            outcome = item.get('outcome')
            if claim_id is None or outcome is None:
                continue
            if claim_id not in tally:
                tally[claim_id] = {}
                order.append(claim_id)
            tally[claim_id][outcome] = tally[claim_id].get(outcome, 0) + 1
    consensus = []
    for claim_id in order:
        counts = tally[claim_id]
        top_n = max(counts.values())
        winners = [oc for oc in counts if counts[oc] == top_n]
        pick = min(winners, key=lambda oc: abs(a_share.get(oc, 0.5) - 0.65))
        consensus.append({'claim_id': claim_id, 'outcome': pick})
    return consensus


MODEL_CONTEXT_LIMIT = MAX_MODEL_LEN
RESOLVED_GENERATION_CONFIG = {
    'engine': 'vllm-local',
    'structured_outputs': USE_STRUCTURED_OUTPUTS,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'enable_thinking': ENABLE_THINKING,
    'thinking_by_stage': {
        'issue_mapper': False,
        'legal_judge_vote': ENABLE_THINKING,
        'direct_fallback': False,
    },
    'sampling': SAMPLING,
    'judge_votes': JUDGE_VOTES,
    'max_new_tokens_by_stage': {
        'issue_mapper': MAX_NEW_TOKENS_ISSUE,
        'legal_judge_vote': MAX_NEW_TOKENS_JUDGE,
        'direct_fallback': MAX_NEW_TOKENS_DIRECT,
    },
}

LABEL_GUIDE = '''Nhãn cuối (xét theo GIÁ TRỊ phần MATERIAL, không đếm số claim):
- A_WIN: gần như TOÀN BỘ yêu cầu của A được chấp nhận. Chỉ được bỏ qua khác biệt thuần thủ tục (án phí, chi phí tố tụng, làm tròn số lẻ).
  Nếu toà cắt một khoản tiền có giá trị mà A đòi — KỂ CẢ tiền lãi hay tiền phạt — thì đó là PARTIAL_A_WIN chứ KHÔNG phải A_WIN.
- B_WIN: mọi hoặc gần như mọi phần MATERIAL của A bị bác.
- PARTIAL_A_WIN: có ít nhất một phần MATERIAL được chấp nhận VÀ một phần MATERIAL bị bác hoặc bị cắt đáng kể, trong đó A giữ phần lớn giá trị MATERIAL.
- PARTIAL_B_WIN: cũng chia như trên nhưng B giữ phần lớn giá trị MATERIAL.
Chỉ dùng PARTIAL khi chỉ ra được phần MATERIAL cụ thể mà A không đạt được.'''

GROUNDING_RULES = f'''Chỉ dùng CASE_QUERY, THÔNG TIN VỤ VIỆC (evidence, nếu có) và {TOP_K} điều luật truy xuất; A là nguyên đơn, B là bị đơn.
Xem các tình tiết được kể trong CASE_QUERY và THÔNG TIN VỤ VIỆC là dữ kiện, NHƯNG mức tiền/định lượng mà A yêu cầu chỉ là ĐÒI HỎI phải được xét, KHÔNG phải phần toà đã chấp nhận.
Thông tin không xuất hiện là UNKNOWN: không dùng sự im lặng để tự động ACCEPT hoặc REJECT, và không đòi thêm tài liệu chỉ vì bản tóm tắt không liệt kê.
Thực tế xét xử dân sự Việt Nam: toà tách TRÁCH NHIỆM (B có phải bồi thường/thực hiện nghĩa vụ không) khỏi ĐỊNH LƯỢNG (mức tiền cụ thể), và cách xử lý ĐỊNH LƯỢNG phụ thuộc LOẠI yêu cầu:
- Khoản tiền do A tự ước tính (bồi thường thiệt hại, chi phí, tổn thất tinh thần): ngay cả khi trách nhiệm của B rõ, toà RẤT THƯỜNG chỉ chấp nhận MỘT PHẦN (giảm khoản thiếu chứng từ, lỗi hỗn hợp, khấu trừ nghĩa vụ đối ứng).
- Khoản tiền có căn cứ tính toán xác định (nợ gốc theo hợp đồng tín dụng/giấy vay, số tiền bị đơn đã thừa nhận) và các yêu cầu KHÔNG chia nhỏ được (đòi lại đúng thửa đất/tài sản, hủy hoặc công nhận một giao dịch, tiếp tục thực hiện hợp đồng): toà thường chấp nhận TOÀN BỘ hoặc bác TOÀN BỘ, chứ không cắt đôi.
Vì vậy không được áp một mặc định duy nhất cho mọi yêu cầu; phải xét theo loại yêu cầu.'''

# Model chưa bao giờ được nhìn thấy schema: USE_STRUCTURED_OUTPUTS=False nên không có
# grammar ép, mà prompt cũ chỉ nói "Chỉ trả JSON theo schema" — hệ quả là model tự đặt tên
# khóa (reliefs/issue_map/claim_type/...) và validate_issue_map vứt sạch => mapper_fallback
# 50/50 ở mọi lần chạy. Khối dưới in ĐÚNG tên khóa + enum cho model.
ISSUE_MAPPER_SCHEMA_BLOCK = """ĐỊNH DẠNG BẮT BUỘC — in ĐÚNG một object JSON, dùng CHÍNH XÁC các tên khóa sau, không đổi tên, không thêm khóa nào khác:
{
  "claims": [
    {
      "claim_id": "C1",
      "request_quote": "<đoạn copy nguyên văn liên tục từ CASE_QUERY>",
      "request_type": "PRINCIPAL",
      "importance": "MATERIAL"
    }
  ]
}
Mỗi claim BẮT BUỘC đủ 4 khóa đúng tên: claim_id, request_quote, request_type, importance.
"request_type" là LOẠI yêu cầu (KHÔNG phải mức quan trọng), chỉ nhận đúng một trong:
- PRINCIPAL: tiền gốc, nợ gốc, tiền vay/hụi/phường/công nợ phải trả.
- INTEREST: lãi, lãi suất, lãi chậm trả.
- PENALTY: phạt vi phạm, phạt cọc, án phí, lệ phí.
- DAMAGES: bồi thường thiệt hại, chi phí điều trị/sửa chữa, tổn thất tinh thần, thu nhập bị mất.
- PROPERTY: đòi lại hoặc giao trả ĐÚNG một tài sản/thửa đất cụ thể, công nhận quyền sử dụng, xử lý/phát mại tài sản thế chấp, hủy giấy chứng nhận.
- DIVISION: yêu cầu CHIA được theo tỷ lệ hoặc theo phần — chia di sản/thừa kế, chia tài sản chung, xác định lại ranh giới hoặc diện tích.
- TRANSACTION: hủy, tuyên vô hiệu, công nhận, hoặc buộc tiếp tục thực hiện hợp đồng/giao dịch.
- OTHER: yêu cầu không thuộc các nhóm trên.
"importance" chỉ nhận MATERIAL hoặc SECONDARY.
Ví dụ output hợp lệ:
{"claims":[{"claim_id":"C1","request_quote":"buộc trả nợ gốc 500.000.000 đồng","request_type":"PRINCIPAL","importance":"MATERIAL"},{"claim_id":"C2","request_quote":"tiền lãi theo hợp đồng","request_type":"INTEREST","importance":"SECONDARY"}]}
Không in giải thích, không in reasoning, không in markdown, không in thẻ think."""

ISSUE_MAPPER_SYSTEM_PROMPT = f'''Bạn là Issue Mapper trung lập.
Chỉ inventory relief mà A thực sự yêu cầu trong CASE_QUERY. Không tạo legal issue, defense hay counterclaim thành claim.
Tách RIÊNG từng khoản có thể được toà quyết định độc lập: tiền gốc, lãi, phạt, TỪNG khoản thiệt hại/chi phí khác nhau, hủy/công nhận giao dịch, trả tài sản. Ví dụ "chi phí sửa xe" và "viện phí" là HAI claim tách biệt. Khi CASE_QUERY nối nhiều khoản bằng "và" hoặc dấu phẩy, hãy tách thành nhiều claim.
request_quote phải là đoạn liên tục copy từ CASE_QUERY (mỗi claim trích đúng phần của mình).
TRƯỚC KHI IN, rà lại CASE_QUERY một lượt từ đầu đến cuối: mỗi động từ yêu cầu (yêu cầu / buộc / đề nghị / xin / đòi) và mỗi khoản tiền riêng biệt được nhắc tới phải nằm trong đúng một claim. Nếu A đòi cả nợ gốc lẫn lãi, đó là HAI claim. Nếu A đòi một tài sản và kèm khoản tiền thay thế khi không giao được, đó là HAI claim.
Ngược lại, nếu vụ việc thật sự chỉ có một yêu cầu duy nhất thì trả đúng một claim — tuyệt đối không bịa thêm relief không có trong CASE_QUERY.
MATERIAL gồm tiền gốc, quyền/tài sản/giao dịch chính, trả tài sản, hủy hoặc công nhận giao dịch.
SECONDARY gồm lãi, phạt, án phí và chi phí phụ.
{GROUNDING_RULES}
{ISSUE_MAPPER_SCHEMA_BLOCK}'''

JUDGE_SYSTEM_PROMPT = f'''Bạn là Legal Judge dự đoán kết quả tranh chấp dân sự Việt Nam.
Đọc kỹ CASE_QUERY và luật; issue map chỉ là inventory hỗ trợ, có thể chưa hoàn hảo.
BẮT BUỘC cân nhắc hai chiều trước khi quyết mỗi claim: (1) căn cứ MẠNH NHẤT để A được chấp nhận; (2) căn cứ MẠNH NHẤT để toà GIẢM hoặc BÁC một phần (thiếu chứng từ chứng minh mức tiền, lỗi hỗn hợp, mức đòi cao hơn thiệt hại thực, nghĩa vụ đối ứng của A). Chỉ FULL_ACCEPT khi CẢ trách nhiệm LẪN mức yêu cầu đều gần như chắc chắn.
Mỗi claim dùng một outcome (phải trung thực vì nhãn cuối được suy ra từ chúng):
- FULL_ACCEPT: A gần như chắc chắn được chấp nhận TOÀN BỘ claim (cả trách nhiệm và mức tiền).
- PARTIAL_A_LEAN: A thắng phần lớn nhưng khả năng bị giảm/không toàn bộ (điển hình cho yêu cầu bồi thường bằng tiền còn tranh cãi về mức).
- PARTIAL_B_LEAN: claim bị chia nhưng B giữ phần lớn hơn.
- REJECT: A gần như bị bác toàn bộ claim.
Với MỖI claim phải cân nhắc ĐỦ BA khả năng (chấp nhận toàn bộ / chấp nhận một phần / bác) rồi mới chọn; không ép về nhị phân và cũng không mặc định PARTIAL.
Chọn outcome theo request_type của claim:
- DAMAGES: mức tiền do A tự ước tính. Trách nhiệm rõ nhưng mức còn tranh cãi -> PARTIAL_A_LEAN; không có căn cứ trách nhiệm -> REJECT.
- PRINCIPAL, INTEREST: số tiền tính được từ hợp đồng/giấy vay/sao kê. Nếu nghĩa vụ và cách tính có căn cứ -> FULL_ACCEPT (đừng hạ xuống PARTIAL chỉ vì con số lớn); nếu khoản nợ bị phản bác hoặc không chứng minh được -> REJECT; chỉ PARTIAL khi chỉ ra được phần nào được thừa nhận và phần nào bị bác.
- PROPERTY, TRANSACTION: relief nguyên khối (đòi lại đúng một thửa đất/tài sản, hủy hoặc công nhận một giao dịch cụ thể) thường KHÔNG cắt đôi được -> FULL_ACCEPT hoặc REJECT tuỳ bên nào có căn cứ pháp lý mạnh hơn (giấy chứng nhận, hợp đồng công chứng, thời hiệu, người thứ ba ngay tình, thực tế quản lý). Nhưng nếu toà có thể công nhận MỘT PHẦN diện tích/giá trị thì vẫn chọn PARTIAL.
- DIVISION: relief CHIA ĐƯỢC (chia di sản/thừa kế, chia tài sản chung, xác định lại ranh giới/diện tích). Ở nhóm này PARTIAL là kết quả BÌNH THƯỜNG NHẤT: toà hiếm khi chia đúng y nguyên tỷ lệ A đề nghị và cũng hiếm khi bác sạch. Chỉ REJECT khi A hoàn toàn không có quyền hưởng, chỉ FULL_ACCEPT khi phương án chia của A được giữ nguyên.
- PENALTY và các khoản phụ: toà hay điều chỉnh, PARTIAL là bình thường.
PARTIAL có HAI chiều, phải dùng cả hai: PARTIAL_A_LEAN khi phần A đạt được LỚN HƠN phần bị bác; PARTIAL_B_LEAN khi phần A đạt được NHỎ HƠN phần bị bác (ví dụ A đòi 10 phần chỉ được 3, hoặc A thắng về nguyên tắc nhưng mất phần lớn giá trị). Đừng mặc định mọi PARTIAL đều là PARTIAL_A_LEAN.
Chọn PARTIAL_A_LEAN hay PARTIAL_B_LEAN thì phải nêu được cụ thể phần nào được chấp nhận và phần nào bị bác; nếu không nêu được thì đó không phải PARTIAL.
Không kết luận FULL_ACCEPT chỉ vì thấy trách nhiệm của B mà chưa xét mức yêu cầu. Không kết luận REJECT chỉ vì bản tóm tắt thiếu tài liệu.
TRƯỚC KHI chốt prediction là A_WIN hoặc B_WIN, kiểm tra lại một lần: có khoản nào A đòi mà bị cắt, bị giảm hoặc bị bác không, và có khoản nào A vẫn giữ được không? Nếu có cả hai thì nhãn đúng là PARTIAL_A_WIN hoặc PARTIAL_B_WIN.
Ưu tiên điều luật khớp trực tiếp tình tiết; đánh giá theo GIÁ TRỊ phần MATERIAL, không đếm claim thô.
{LABEL_GUIDE}
{GROUNDING_RULES}
Trả JSON theo schema, chỉ in một object JSON với đúng các khóa: claim_outcomes (mảng, mỗi phần tử có claim_id, outcome, legal_basis_ranks), prediction (nhãn tổng thể nhất quán với các outcome), confidence (0..1).'''

DIRECT_SYSTEM_PROMPT = f'''Bạn là bộ phân loại dự phòng cho tranh chấp dân sự Việt Nam.
Đọc CASE_QUERY và {TOP_K} điều luật rồi chọn đúng một nhãn. Không dùng gold label hoặc dữ liệu đánh giá.
{LABEL_GUIDE}
{GROUNDING_RULES}
Chỉ trả compact JSON theo schema.'''


def build_law_context(laws):
    blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f'{content}'
        )
    return '\n\n'.join(blocks)


def build_issue_prompt(case_query, laws):
    return f'''CASE_QUERY:
{case_query.strip()}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Inventory các relief của A theo thứ tự xuất hiện.'''


def build_judge_prompt(case_query, laws, issue_map, case_facts=''):
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

ISSUE MAP:
{json.dumps(issue_map, ensure_ascii=False, separators=(',', ':'))}

Quyết định từng claim và prediction tổng thể. prediction phải phản ánh giá trị phần MATERIAL, không phải số claim.'''


def build_direct_prompt(case_query, laws, case_facts=''):
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Chọn nhãn dự đoán cuối cùng.'''


def build_messages(system_prompt, user_prompt):
    # Ton trong khuyen nghi vendor: model dat use_system_prompt=False (vd DeepSeek-R1)
    # gop system vao user turn. Noi dung prompt GIU NGUYEN, chi doi cach dong goi message.
    if ACTIVE_PROFILE['use_system_prompt']:
        return [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
    return [{'role': 'user', 'content': system_prompt + '\n\n' + user_prompt}]


def count_message_tokens(messages):
    # Qwen: giu nguyen (enable_thinking=False nhu cu). Model khac: bo kwarg de tokenizer khong ken.
    _kw = {'enable_thinking': False} if 'qwen' in MODEL_ID.lower() else {}
    return len(prompt_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        **_kw,
    ))


def _normalize_match(text):
    normalized = unicodedata.normalize('NFKC', str(text or '')).casefold()
    return re.sub(r'[\W_]+', ' ', normalized, flags=re.UNICODE).strip()


# Dong tu/danh tu chi YEU CAU. Da bo cac tu qua ngan de tranh dinh nham sau khi khu
# dau: 'doi' dung mot minh trung voi 'doi' trong 'doi voi', 'nhan' trung 'nhan dan'.
_RELIEF_MARKERS = (
    'yeu cau', 'buoc', 'de nghi', 'doi lai', 'doi boi thuong', 'chia', 'tra', 'tra lai',
    'tra no', 'hoan tra', 'thanh toan', 'boi thuong', 'cong nhan', 'huy', 'huy bo',
    'giao', 'giao tra', 'lai', 'an phi', 'tiep tuc thuc hien', 'cham dut', 'thu hoi',
    'den bu', 'xac dinh lai', 'cap duong',
)
# Cum mo ta DOI TUONG tranh chap, khong phai relief -> khong duoc thanh claim.
_TOPIC_MARKERS = ('tranh chap', 'vu an', 'quan he phap luat')


def _is_query_span(candidate, case_query):
    """Chan hallucinate, nhung khong phat model vi dien dat lai dong tu dan.

    Do tren raw output lan chay truoc: 6/115 claim bi vut, trong do 4 la relief CO
    THAT chi khac cach dan (model viet 'buoc bi don giao nha dat' trong khi query
    viet 'yeu cau bi don giao nha dat'), 2 la cau mo ta doi tuong tranh chap (dung
    ra phai vut). Nen: van uu tien trich nguyen van; neu khong, chi chap nhan khi
    >=80% tu cua quote co mat trong query VA quote thuc su chua tu chi yeu cau VA
    khong phai cum mo ta tranh chap. Da test tren raw output that: 4 relief that
    duoc cuu, 2 cum mo ta van bi loai, moi quote bia dat deu bi tu choi.
    """
    candidate_norm = _normalize_match(candidate)
    query_norm = _normalize_match(case_query)
    if not candidate_norm:
        return False
    if candidate_norm in query_norm:
        return True
    padded = ' ' + _strip_accents(candidate_norm) + ' '
    if any((' ' + marker + ' ') in padded for marker in _TOPIC_MARKERS):
        return False
    tokens = candidate_norm.split()
    if len(tokens) < 2:
        return False
    query_tokens = set(query_norm.split())
    coverage = sum(1 for token in tokens if token in query_tokens) / len(tokens)
    has_relief = any((' ' + marker + ' ') in padded for marker in _RELIEF_MARKERS)
    return coverage >= 0.8 and has_relief


def fallback_issue_map(case_query, reason):
    return {
        'claims': [{
            'claim_id': 'C1',
            'request_quote': case_query.strip(),
            'request_type': 'OTHER',
            'importance': 'MATERIAL',
        }],
        'mapper_fallback': True,
        'mapper_fallback_reason': str(reason),
    }


VALID_REQUEST_TYPES = ('PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES',
                       'PROPERTY', 'DIVISION', 'TRANSACTION', 'OTHER')
# FIX: INTEREST khong con tu dong SECONDARY. Trong tranh chap tin dung, lai la mot
# phan gia tri lon va viec toa cat lai chinh la thu bien vu an thanh PARTIAL_A_WIN;
# xep no la SECONDARY khien nhan bi keo len A_WIN (v11: case_6663, case_3241).
SECONDARY_BY_TYPE = {'PENALTY'}

# Tên khóa model hay dùng thay cho tên trong schema (đo trên raw output lần chạy trước).
_QUOTE_KEYS = ('request_quote', 'quote', 'relief_quote', 'claim_quote', 'request',
               'specific_relief', 'claim', 'issue')
_TYPE_KEYS = ('request_type', 'claim_type', 'type', 'claim_category', 'category',
              'issue_type', 'claim_subtype', 'sub_category', 'classification')
_IMPORTANCE_KEYS = ('importance', 'materiality', 'importance_level', 'material',
                    'material_claim', 'secondary', 'secondary_material')
_TYPE_ALIASES = {
    'PRINCIPAL': 'PRINCIPAL', 'DEBT': 'PRINCIPAL', 'LOAN': 'PRINCIPAL', 'MONEY': 'PRINCIPAL',
    'GOC': 'PRINCIPAL', 'NO_GOC': 'PRINCIPAL', 'PAYMENT': 'PRINCIPAL',
    'INTEREST': 'INTEREST', 'LAI': 'INTEREST', 'LAI_SUAT': 'INTEREST',
    'PENALTY': 'PENALTY', 'FINE': 'PENALTY', 'PHAT': 'PENALTY', 'FEE': 'PENALTY',
    'DAMAGES': 'DAMAGES', 'DAMAGE': 'DAMAGES', 'COMPENSATION': 'DAMAGES',
    'BOI_THUONG': 'DAMAGES', 'THIET_HAI': 'DAMAGES', 'COST': 'DAMAGES',
    'PROPERTY': 'PROPERTY', 'ASSET': 'PROPERTY', 'LAND': 'PROPERTY', 'REAL_ESTATE': 'PROPERTY',
    'OWNERSHIP': 'PROPERTY', 'TAI_SAN': 'PROPERTY', 'DAT': 'PROPERTY',
    'DIVISION': 'DIVISION', 'INHERITANCE': 'DIVISION', 'PARTITION': 'DIVISION',
    'CHIA': 'DIVISION', 'THUA_KE': 'DIVISION', 'DI_SAN': 'DIVISION', 'BOUNDARY': 'DIVISION',
    'TRANSACTION': 'TRANSACTION', 'CONTRACT': 'TRANSACTION', 'AGREEMENT': 'TRANSACTION',
    'NULLIFICATION': 'TRANSACTION', 'HOP_DONG': 'TRANSACTION', 'GIAO_DICH': 'TRANSACTION',
    'OTHER': 'OTHER', 'MISC': 'OTHER', 'KHAC': 'OTHER',
}
# Backstop khi model không cho enum dùng được: suy request_type từ chính đoạn trích.
_TYPE_PATTERNS = (
    ('INTEREST', r'(lai suat|tien lai|lai trong han|lai qua han|lai cham tra|lai phat sinh|goc va lai|va lai)'),
    ('PENALTY', r'(phat vi pham|tien phat|phat coc|phat hop dong|phat cham|an phi|le phi)'),
    ('DAMAGES', r'(boi thuong|thiet hai|chi phi dieu tri|chi phi chua|vien phi|ton that tinh than|chi phi sua|mai tang|cap duong|thu nhap bi mat|cong suc)'),
    ('TRANSACTION', r'(huy (bo )?(hop dong|giao dich|van ban|thoa thuan)|vo hieu|cong nhan (hop dong|giao dich|hieu luc)|tiep tuc thuc hien|cham dut hop dong|chuyen nhuong)'),
    ('DIVISION', r'(chia (deu |doi |theo )?(di san|thua ke|tai san|gia tri|phan)|phan chia|thua ke theo phap luat|di san thua ke|tai san chung|ranh gioi|dien tich thuc te)'),
    ('PROPERTY', r'(quyen su dung dat|su dung dat|tra (lai )?(nha|dat|tai san|phan dat|dien tich)|giao (tra|lai )?dat|thua ke|di san|chia (deu )?(tai san|di san|gia tri)|so do|giay chung nhan|thao do|di doi|lan chiem|so huu|phat mai|tai san the chap|xu ly tai san)'),
    ('PRINCIPAL', r'(no goc|von goc|tien vay|tien hui|tien phuong|cong no|tra no|thanh toan|hoan tra|lien doi tra|buoc tra|nghia vu tra)'),
)


def _strip_accents(text):
    decomposed = unicodedata.normalize('NFD', str(text or '').casefold())
    return ''.join(ch for ch in decomposed if unicodedata.category(ch) != 'Mn').replace('đ', 'd')


def _first_list_of_dicts(obj, depth=0):
    """Model hay đặt danh sách claim dưới tên khóa khác (reliefs/issue_map/...).
    Nhận danh sách dict dài nhất; nếu object chính là một claim đơn thì bọc lại."""
    if not isinstance(obj, dict) or depth > 2:
        return None
    lists = [v for v in obj.values()
             if isinstance(v, list) and v and all(isinstance(i, dict) for i in v)]
    if lists:
        return max(lists, key=len)
    if any(k in obj for k in _QUOTE_KEYS):
        return [obj]
    for value in obj.values():
        nested = _first_list_of_dicts(value, depth + 1)
        if nested:
            return nested
    return None


def _pick_field(raw, keys):
    for key in keys:
        if key in raw and raw[key] not in (None, ''):
            return raw[key]
    return None


def _infer_type_from_quote(quote):
    if re.search(r'lãi', str(quote or ''), flags=re.I):
        return 'INTEREST'
    flat = re.sub(r'[\W_]+', ' ', _strip_accents(quote)).strip()
    for name, pattern in _TYPE_PATTERNS:
        if re.search(pattern, flat):
            return name
    return 'OTHER'


# Relief CHIA DUOC bi model gan nham sang PROPERTY/TRANSACTION thi se an chinh sach
# all-or-nothing va sup ve A_WIN/B_WIN (v11: case_614, case_7467). Dau hieu "chia di san /
# chia thua ke / chia tai san chung / ranh gioi" la xac dinh du de ghi de.
_DIVISION_OVERRIDE = re.compile(
    r'(chia (deu |doi |theo )?(di san|thua ke|tai san|gia tri|phan)|phan chia'
    r'|thua ke theo phap luat|di san thua ke|tai san chung|ranh gioi|dien tich thuc te)'
)


def _pick_request_type(raw, quote):
    if _DIVISION_OVERRIDE.search(re.sub(r'[\W_]+', ' ', _strip_accents(quote)).strip()):
        return 'DIVISION'
    value = _pick_field(raw, _TYPE_KEYS)
    token = re.sub(r'[\W]+', '_', str(value or '').strip().upper()).strip('_')
    # Model rất hay nhét MATERIAL/SECONDARY vào ô type -> đó là importance, không phải type.
    if token and token not in {'MATERIAL', 'SECONDARY', 'PRIMARY'}:
        if token in VALID_REQUEST_TYPES:
            return token
        if token in _TYPE_ALIASES:
            return _TYPE_ALIASES[token]
        for alias, canonical in _TYPE_ALIASES.items():
            if alias in token:
                return canonical
    return _infer_type_from_quote(quote)


def _pick_importance(raw, request_type):
    if request_type in SECONDARY_BY_TYPE:
        return 'SECONDARY'
    # Model rat hay dao cho: nhet MATERIAL/SECONDARY vao o type. Van la tin hieu importance.
    for key in _TYPE_KEYS:
        token = str(raw.get(key, '') or '').strip().upper()
        if token in {'MATERIAL', 'PRIMARY'}:
            return 'MATERIAL'
        if token == 'SECONDARY':
            return 'SECONDARY'
    for key in _IMPORTANCE_KEYS:
        if key not in raw or raw[key] in (None, ''):
            continue
        value = raw[key]
        if isinstance(value, bool):
            if key == 'secondary':
                return 'SECONDARY' if value else 'MATERIAL'
            return 'MATERIAL' if value else 'SECONDARY'
        token = str(value).strip().upper()
        if 'SECOND' in token or token in {'PHU', 'FALSE', 'NO', 'LOW'}:
            return 'SECONDARY'
        if 'MATERIAL' in token or 'CHINH' in token or token in {'TRUE', 'YES', 'MAIN', 'PRIMARY', 'HIGH'}:
            return 'MATERIAL'
    return 'MATERIAL'


def validate_issue_map(obj, case_query):
    """Chấp nhận tên khóa lệch schema. Grounding vẫn bắt buộc qua _is_query_span:
    ưu tiên trích nguyên văn, chỉ nới đúng mức cho trường hợp model đổi động từ dẫn
    ('buộc' thay cho 'yêu cầu'). Quote bịa đặt vẫn bị từ chối — đây vẫn là chốt
    chống hallucinate, không được nới thêm."""
    raw_claims = obj.get('claims')
    if not isinstance(raw_claims, list) or not raw_claims:
        raw_claims = _first_list_of_dicts(obj)
    if not isinstance(raw_claims, list):
        raise ValueError('claims must be a list')
    claims, seen = [], set()
    for raw in raw_claims:
        if not isinstance(raw, dict):
            continue
        quote = str(_pick_field(raw, _QUOTE_KEYS) or '').strip()
        if not _is_query_span(quote, case_query):
            continue
        request_type = _pick_request_type(raw, quote)
        importance = _pick_importance(raw, request_type)
        key = (_normalize_match(quote), request_type)
        if key in seen:
            continue
        seen.add(key)
        claims.append({
            'claim_id': f'C{len(claims) + 1}',
            'request_quote': quote,
            'request_type': request_type,
            'importance': importance,
        })
        if len(claims) >= 8:  # khớp maxItems của ISSUE_SCHEMA
            break
    if not claims:
        raise ValueError('no grounded unique claims survived validation')
    if not any(claim['importance'] == 'MATERIAL' for claim in claims):
        claims[0]['importance'] = 'MATERIAL'
    return {'claims': claims, 'mapper_fallback': False, 'mapper_fallback_reason': None}


def _json_object(text):
    cleaned = (text or '')
    # Thinking mode phát ra <think>...</think> trước JSON; bỏ đi như base.
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.I | re.S)
    if '</think>' in cleaned:
        cleaned = cleaned.rsplit('</think>', 1)[-1]
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', cleaned.strip(), flags=re.I | re.S).strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', cleaned):
        try:
            obj, _ = decoder.raw_decode(cleaned[match.start():])
        except Exception:
            continue
        if isinstance(obj, dict):
            return obj
    raise ValueError('no complete JSON object')


def derive_label(issue_map, claim_outcomes):
    '''Suy nhãn 4 lớp từ outcome của từng claim, trọng số theo importance.
    PADO đúng nghĩa: nhãn tổng thể là hệ quả của các quyết định claim, không phải
    nhãn model tự khai. Đây là điều làm PARTIAL xuất hiện đúng cấu trúc.'''
    weight_by_id = {
        c['claim_id']: (2.0 if c.get('importance') == 'MATERIAL' else 1.0)
        for c in issue_map['claims']
    }
    a_share = {'FULL_ACCEPT': 1.0, 'PARTIAL_A_LEAN': 0.65, 'PARTIAL_B_LEAN': 0.35, 'REJECT': 0.0}
    num = den = 0.0
    for co in claim_outcomes:
        w = weight_by_id.get(co['claim_id'], 1.0)
        num += w * a_share.get(co['outcome'], 0.5)
        den += w
    if den == 0:
        return None
    s = num / den
    if s >= 0.85:
        return 'A_WIN'
    if s >= 0.55:
        return 'PARTIAL_A_WIN'
    if s >= 0.30:
        return 'PARTIAL_B_WIN'
    return 'B_WIN'


def validate_judge(obj, issue_map, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0

    valid_outcomes = {'FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'}
    expected_ids = [claim['claim_id'] for claim in issue_map['claims']]
    raw_by_id = {}
    for item in obj.get('claim_outcomes', []) if isinstance(obj.get('claim_outcomes'), list) else []:
        if not isinstance(item, dict):
            continue
        claim_id = str(item.get('claim_id', '')).strip()
        outcome = str(item.get('outcome', '')).strip().upper()
        if claim_id not in expected_ids or outcome not in valid_outcomes or claim_id in raw_by_id:
            continue
        ranks = []
        for rank in item.get('legal_basis_ranks', []) if isinstance(item.get('legal_basis_ranks'), list) else []:
            try:
                rank = int(rank)
            except Exception:
                continue
            if 1 <= rank <= len(retrieved_laws) and rank not in ranks:
                ranks.append(rank)
        raw_by_id[claim_id] = {
            'claim_id': claim_id,
            'outcome': outcome,
            'legal_basis_ranks': ranks[:3],
        }

    default_outcome = {
        'A_WIN': 'FULL_ACCEPT',
        'PARTIAL_A_WIN': 'PARTIAL_A_LEAN',
        'PARTIAL_B_WIN': 'PARTIAL_B_LEAN',
        'B_WIN': 'REJECT',
    }[prediction]
    outcomes = [
        raw_by_id.get(claim_id, {
            'claim_id': claim_id,
            'outcome': default_outcome,
            'legal_basis_ranks': [],
        })
        for claim_id in expected_ids
    ]
    ranks = []
    for item in outcomes:
        for rank in item['legal_basis_ranks']:
            if rank not in ranks:
                ranks.append(rank)
    applied_laws = [
        {
            'law_id': str(retrieved_laws[rank - 1]['law_id']),
            'aid': int(retrieved_laws[rank - 1]['aid']),
            'reason': f'Legal Judge selected retrieved law rank {rank}',
        }
        for rank in ranks
    ]
    derived = derive_label(issue_map, outcomes)
    # 'prediction' o day van la nhan holistic cua tung phieu - aggregate_votes dung no
    # de chon phieu dai dien. Nhan CUOI CUNG cua case duoc suy tu claim_outcomes o cell
    # ket qua (xem 'decider' trong result). Comment cu khuyen khong dung derived la viet
    # cho thoi mapper hong (1 claim/case, derive suy bien) - dieu kien do khong con dung.
    return {
        'prediction': prediction,
        'derived_label': derived,
        'confidence': confidence,
        'claim_outcomes': outcomes,
        'applied_laws': applied_laws,
    }


def validate_direct(obj):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid direct prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0
    return {'prediction': prediction, 'confidence': confidence}


def aggregate_votes(votes):
    if not votes:
        raise ValueError('cannot aggregate zero votes')
    counts = {label: 0 for label in LABELS}
    confidence_sum = {label: 0.0 for label in LABELS}
    for vote in votes:
        label = vote['prediction']
        counts[label] += 1
        confidence_sum[label] += vote['confidence']
    max_count = max(counts.values())
    candidates = [label for label in LABELS if counts[label] == max_count]
    # Hoa phieu = cac vote khong dong y = bat dinh. Truoc day tie-break la
    # -LABELS.index() nen luon roi ve A_WIN (LABELS[0]) - mot thien lech ve nhan
    # cuc doan dung luc bang chung yeu nhat. Gio hoa phieu thi nghieng ve nhan
    # trung dung (PARTIAL) truoc.
    tie_break_rank = {'PARTIAL_A_WIN': 3, 'PARTIAL_B_WIN': 2, 'A_WIN': 1, 'B_WIN': 0}
    candidates.sort(
        key=lambda label: (
            confidence_sum[label] / max(1, counts[label]),
            tie_break_rank[label],
        ),
        reverse=True,
    )
    prediction = candidates[0]
    selected = [vote for vote in votes if vote['prediction'] == prediction]
    representative = max(selected, key=lambda vote: vote['confidence'])
    return {
        'prediction': prediction,
        'confidence': sum(vote['confidence'] for vote in selected) / len(selected),
        'vote_counts': counts,
        'n_valid_votes': len(votes),
        'claim_outcomes': representative['claim_outcomes'],
        'applied_laws': representative['applied_laws'],
    }


def deterministic_stage_seed(eval_seed, case_id, stage):
    raw = f'{eval_seed}|{case_id}|{stage}'.encode('utf-8')
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'big')


ISSUE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claims': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 8,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
                    'request_quote': {'type': 'string', 'minLength': 1, 'maxLength': 600},
                    'request_type': {
                        'type': 'string',
                        'enum': ['PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES', 'PROPERTY', 'DIVISION', 'TRANSACTION', 'OTHER'],
                    },
                    'importance': {'type': 'string', 'enum': ['MATERIAL', 'SECONDARY']},
                },
                'required': ['claim_id', 'request_quote', 'request_type', 'importance'],
            },
        },
    },
    'required': ['claims'],
}

CLAIM_OUTCOME_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
        'outcome': {
            'type': 'string',
            'enum': ['FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'],
        },
        'legal_basis_ranks': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 3,
            'items': {'type': 'integer', 'minimum': 1, 'maximum': 10},
        },
    },
    'required': ['claim_id', 'outcome', 'legal_basis_ranks'],
}

JUDGE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_outcomes': {'type': 'array', 'minItems': 1, 'maxItems': 8, 'items': CLAIM_OUTCOME_SCHEMA},
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['claim_outcomes', 'prediction', 'confidence'],
}

DIRECT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['prediction', 'confidence'],
}

HARD_FALLBACK_LABEL = os.getenv('ALQAC_HARD_FALLBACK_LABEL', 'PARTIAL_A_WIN').strip().upper()
if HARD_FALLBACK_LABEL not in LABELS:
    raise ValueError(f'Invalid ALQAC_HARD_FALLBACK_LABEL={HARD_FALLBACK_LABEL!r}')

PIPELINE_CONTRACT = {
    'pipeline_version': PADO_PIPELINE_VERSION,
    'stages': ['issue_mapper', 'legal_judge_3vote', 'direct_fallback'],
    'issue_schema': ISSUE_SCHEMA,
    'judge_schema': JUDGE_SCHEMA,
    'direct_schema': DIRECT_SCHEMA,
    'judge_votes': JUDGE_VOTES,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'hard_fallback_label': HARD_FALLBACK_LABEL,
}
PIPELINE_CONTRACT_HASH = hashlib.sha256(
    json.dumps(PIPELINE_CONTRACT, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()


def aggregate_stage_usage(stage_usages):
    values = list(stage_usages.values())
    return {
        'input_tokens': sum(item['input_tokens'] for item in values),
        'output_tokens': sum(item['output_tokens'] for item in values),
        'total_tokens': sum(item['total_tokens'] for item in values),
        'hit_max_new_tokens': any(item.get('hit_max_new_tokens', False) for item in values),
        'stage_count': len(values),
        'stages': stage_usages,
    }


BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'pipeline_version': PADO_PIPELINE_VERSION,
    'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
    'pipeline_stages': PIPELINE_CONTRACT['stages'],
    'model_name': MODEL_NAME,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'generation': RESOLVED_GENERATION_CONFIG,
    'model_context_limit': MODEL_CONTEXT_LIMIT,
    'dtype': MODEL_DTYPE,
    'full_gpu_no_quantization': True,
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases),
    'labels': LABELS,
    'retrieval_source': RETRIEVAL_SOURCE,
    'retrieval_path': str(RETRIEVAL_PATH),
    'retrieval_sha256': hashlib.sha256(RETRIEVAL_PATH.read_bytes()).hexdigest(),
    'top_k_laws': TOP_K,
    'retrieval_signature': retrieval_signature,
    'max_input_tokens': MAX_INPUT_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'eval_seeds': EVAL_SEEDS,
    'coverage_policy': 'mapper_fallback + direct_fallback + explicit hard fallback always emit a label',
    'prompt_hashes': {
        'issue_mapper': hashlib.sha256(ISSUE_MAPPER_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'legal_judge': hashlib.sha256(JUDGE_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'direct_fallback': hashlib.sha256(DIRECT_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    },
}

# ---- Preflight ngan sach token (chay local, truoc khi dot GPU) ----
# Top-14 dai hon top-10 dang ke; kiem tra truoc de khong chet giua chung o checked_messages.
def _worst_issue_map(case_query):
    # Upper bound that: schema cho toi da 8 claim, va validate_issue_map bat moi request_quote
    # phai la doan trich tu case_query -> khong claim nao dai hon ca case_query.
    return {
        'claims': [{'claim_id': f'C{i}', 'request_quote': case_query.strip(),
                    'request_type': 'DAMAGES', 'importance': 'MATERIAL'} for i in range(1, 9)],
        'mapper_fallback': False,
    }


_budget = []
for _case in inference_cases:
    _msgs = build_messages(
        JUDGE_SYSTEM_PROMPT,
        build_judge_prompt(_case['case_query'], retrieval_cache[_case['case_id']],
                           _worst_issue_map(_case['case_query']), _case['case_facts']),
    )
    _budget.append((count_message_tokens(_msgs), _case['case_id']))
_max_in, _worst_case = max(_budget)
_headroom = MODEL_CONTEXT_LIMIT - MAX_NEW_TOKENS_JUDGE - _max_in
print(f'Preflight judge prompt (worst-case issue map): input TB='
      f'{sum(t for t, _ in _budget) // len(_budget)} | MAX={_max_in} ({_worst_case})'
      f' | con du {_headroom} token so voi {MODEL_CONTEXT_LIMIT} - {MAX_NEW_TOKENS_JUDGE}')
if _max_in > MAX_INPUT_TOKENS or _headroom < 0:
    raise RuntimeError(
        f'Ngan sach prompt khong du: input toi da {_max_in} token, gioi han '
        f'{MAX_INPUT_TOKENS}/{MODEL_CONTEXT_LIMIT} voi max_new={MAX_NEW_TOKENS_JUDGE}. '
        'Ha ALQAC_MAX_ARTICLE_CHARS (vd 3000) hoac ALQAC_MAX_EVIDENCE_CHARS roi chay lai cell 2-5.'
    )

print(
    'v12 compact ensemble ready | GPU:', GPU_TYPE,
    '| whitespace disabled: True',
    '| thinking:', ENABLE_THINKING,
    '| sampling:', SAMPLING,
    '| votes:', JUDGE_VOTES,
    '| contract:', PIPELINE_CONTRACT_HASH[:12],
    '| retrieval:', RETRIEVAL_SOURCE,
)


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

(EngineCore pid=462) INFO 08-10 02:49:38 [gpu_model_runner.py:5209] Starting to load model Qwen/Qwen3.5-9B...
(EngineCore pid=462) INFO 08-10 02:49:38 [cuda.py:535] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
(EngineCore pid=462) INFO 08-10 02:49:38 [mm_encoder_attention.py:373] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore pid=462) INFO 08-10 02:49:39 [qwen_gdn_linear_attn.py:228] Using Triton/FLA GDN prefill kernel (requested=auto, head_k_dim=128).
(EngineCore pid=462) INFO 08-10 02:49:43 [cuda.py:476] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=462) INFO 08-10 02:49:43 [flash_attn.py:718] Using FlashAttention version 2
(EngineCore pid=462) INFO 08-10 02:51:21 [weight_utils.py:530] Time spent downloading weights for Qwen/Qwen3.5-9B: 93.281083 seconds
(EngineCore pid=462) INFO 08-10 02:51:21 [weight_utils.py:849] Filesystem type for checkpoin

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:05,  1.75s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:03<00:03,  1.70s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:05<00:01,  1.73s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:06<00:00,  1.57s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:06<00:00,  1.62s/it]
(EngineCore pid=462) 


(EngineCore pid=462) INFO 08-10 02:51:27 [default_loader.py:430] Loading weights took 6.51 seconds
(EngineCore pid=462) INFO 08-10 02:51:29 [gpu_model_runner.py:5306] Model loading took 17.66 GiB memory and 108.915584 seconds
(EngineCore pid=462) INFO 08-10 02:51:29 [interface.py:890] Setting attention block size to 528 tokens to ensure that attention page size is >= mamba page size.
(EngineCore pid=462) INFO 08-10 02:51:29 [interface.py:914] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore pid=462) INFO 08-10 02:51:29 [gpu_model_runner.py:6322] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=462) INFO 08-10 02:51:55 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/ee4332365a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=462) INFO 08-10 02:51:55 [backends.py:1148] Dynamo bytecode t

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|█████████████████| 19/19 [00:03<00:00,  5.05it/s]
Capturing CUDA graphs (decode, FULL): 100%|████████████████████████████████████| 11/11 [00:02<00:00,  4.13it/s]


(EngineCore pid=462) INFO 08-10 02:56:00 [gpu_model_runner.py:6707] Graph capturing finished in 8 secs, took 0.19 GiB
(EngineCore pid=462) INFO 08-10 02:56:00 [gpu_worker.py:771] CUDA graph pool memory: 0.19 GiB (actual), 0.23 GiB (estimated), difference: 0.04 GiB (22.4%).
(EngineCore pid=462) INFO 08-10 02:56:00 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=462) INFO 08-10 02:56:01 [core.py:337] init engine (profile, create kv cache, warmup model) took 272.58 s (compilation: 104.97 s)
(EngineCore pid=462) INFO 08-10 02:56:01 [vllm.py:1042] Asynchronous scheduling is enabled.
(EngineCore pid=462) INFO 08-10 02:56:01 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
vLLM ready (local, cung tien trinh voi notebook).
Preflight judge prompt (worst-case issue map): input TB=2 | MAX=2 (case_950) | con du 24574 token so 

In [7]:
def vllm_generate(messages_list, seeds, schema, max_tokens, enable_thinking=None):
    if len(messages_list) != len(seeds):
        raise ValueError('messages_list and seeds length mismatch')
    from vllm import SamplingParams
    try:
        from vllm.sampling_params import StructuredOutputsParams
    except ImportError:
        from vllm import StructuredOutputsParams

    structured = None
    if USE_STRUCTURED_OUTPUTS and schema is not None:
        structured = StructuredOutputsParams(
            json=schema,
            disable_any_whitespace=True,
        )

    def make_params(seed):
        kwargs = dict(max_tokens=max_tokens, seed=int(seed))
        for _k in ('temperature', 'top_p', 'top_k', 'min_p', 'repetition_penalty'):
            if _k in SAMPLING:
                kwargs[_k] = SAMPLING[_k]
        if structured is not None:
            kwargs['structured_outputs'] = structured
        return SamplingParams(**kwargs)

    tmpl_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        tmpl_kwargs['enable_thinking'] = ENABLE_THINKING if enable_thinking is None else enable_thinking
    outputs = vllm_engine.chat(
        messages_list,
        [make_params(seed) for seed in seeds],
        add_generation_prompt=True,
        chat_template_kwargs=tmpl_kwargs,
        use_tqdm=True,
    )
    results = []
    for output in outputs:
        generated = output.outputs[0]
        results.append((generated.text, {
            'input_tokens': len(output.prompt_token_ids),
            'output_tokens': len(generated.token_ids),
            'total_tokens': len(output.prompt_token_ids) + len(generated.token_ids),
            'hit_max_new_tokens': generated.finish_reason == 'length',
            'finish_reason': generated.finish_reason,
        }))
    return results


def close_modal():
    """Giai phong VRAM local."""
    global vllm_engine
    try:
        del vllm_engine
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Da giai phong GPU local.')


In [8]:
_cid = sample_case['case_id']
_query = sample_case['case_query']
_facts = evidence_by_case.get(_cid, '')
_seed = EVAL_SEEDS[0]
_laws = retrieval_cache[_cid]
try:
    mapper_raw, mapper_usage = vllm_generate(
        [build_messages(ISSUE_MAPPER_SYSTEM_PROMPT, build_issue_prompt(_query, _laws))],
        [deterministic_stage_seed(_seed, _cid, 'issue_mapper')],
        ISSUE_SCHEMA,
        MAX_NEW_TOKENS_ISSUE,
        enable_thinking=False,  # khớp stage mapper của lần chạy thật
    )[0]
    try:
        issue_map = validate_issue_map(_json_object(mapper_raw), _query)
    except Exception as mapper_exc:
        issue_map = fallback_issue_map(_query, mapper_exc)

    vote_raw, vote_usage = vllm_generate(
        [build_messages(JUDGE_SYSTEM_PROMPT, build_judge_prompt(_query, _laws, issue_map, _facts))],
        [deterministic_stage_seed(_seed, _cid, 'legal_judge_vote_0')],
        JUDGE_SCHEMA,
        MAX_NEW_TOKENS_JUDGE,
    )[0]
    vote = validate_judge(_json_object(vote_raw), issue_map, _laws)
    print(
        'SMOKE OK | mapper_fallback:', issue_map['mapper_fallback'],
        '| claims:', len(issue_map['claims']),
        '| types:', [c['request_type'] for c in issue_map['claims']],
        '| prediction:', vote['prediction'],
        '| usage:', aggregate_stage_usage({'issue_mapper': mapper_usage, 'legal_judge_vote_0': vote_usage}),
    )
except Exception as exc:
    print('SMOKE WARNING:', repr(exc))


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 08-10 02:56:19 [hf.py:548] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|█| 1/1 [00:01<00:00,  1.68s/it, est. speed input: 4147.13 toks/s, output: 38.23 toks/s]


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|             | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=462) WARNING 08-10 02:56:21 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _zero_kv_blocks_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██| 1/1 [02:22<00:00, 142.34s/it, est. speed input: 65.58 toks/s, output: 68.63 toks/s]

SMOKE OK | mapper_fallback: False | claims: 2 | types: ['DAMAGES', 'DAMAGES'] | prediction: PARTIAL_A_WIN | usage: {'input_tokens': 16276, 'output_tokens': 9832, 'total_tokens': 26108, 'hit_max_new_tokens': False, 'stage_count': 2, 'stages': {'issue_mapper': {'input_tokens': 6942, 'output_tokens': 64, 'total_tokens': 7006, 'hit_max_new_tokens': False, 'finish_reason': 'stop'}, 'legal_judge_vote_0': {'input_tokens': 9334, 'output_tokens': 9768, 'total_tokens': 19102, 'hit_max_new_tokens': False, 'finish_reason': 'stop'}}}


In [9]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')


def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default


def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)


def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'model': model,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'generation': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'],
        'case_facts': case.get('case_facts', ''),
        'laws': laws,
        'prompts': {
            'issue_mapper': ISSUE_MAPPER_SYSTEM_PROMPT,
            'legal_judge': JUDGE_SYSTEM_PROMPT,
            'direct_fallback': DIRECT_SYSTEM_PROMPT,
        },
        'schemas': {'issue': ISSUE_SCHEMA, 'judge': JUDGE_SCHEMA, 'direct': DIRECT_SCHEMA},
    }
    return hashlib.sha256(
        json.dumps(material, ensure_ascii=False, sort_keys=True).encode('utf-8')
    ).hexdigest()


manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

MODEL = MODELS_TO_RUN[0]
EVAL_SEED = EVAL_SEEDS[0]
result_path = OUTPUT_DIR / f'predictions_{slugify(MODEL)}_seed-{EVAL_SEED}.json'
saved = load_json(result_path, {})

cache_keys, cases_run = {}, []
for case in inference_cases:
    cid = case['case_id']
    cache_key = make_cache_key(MODEL, case, retrieval_cache[cid], EVAL_SEED)
    cache_keys[cid] = cache_key
    old = saved.get(cid)
    if old and old.get('cache_key') == cache_key and old.get('prediction') in LABELS:
        continue
    cases_run.append(case)
print(f'=== v12 compact ensemble | seed={EVAL_SEED} | run {len(cases_run)}/{len(inference_cases)} cases ===')

state = {}
for case in cases_run:
    state[case['case_id']] = {
        'laws': retrieval_cache[case['case_id']],
        'raw': {},
        'usage': {},
        'issue_map': None,
        'votes': [],
        'stage_errors': [],
    }


def checked_messages(system_prompt, user_prompt, max_tokens):
    messages = build_messages(system_prompt, user_prompt)
    input_tokens = count_message_tokens(messages)
    if input_tokens > MAX_INPUT_TOKENS or input_tokens + max_tokens > MODEL_CONTEXT_LIMIT:
        raise ValueError(
            f'prompt budget exceeded: input={input_tokens}, output={max_tokens}, '
            f'limits={MAX_INPUT_TOKENS}/{MODEL_CONTEXT_LIMIT}'
        )
    return messages


started_all = time.time()

# Stage 1: mapper. Every failure is converted to a one-claim grounded fallback.
mapper_messages, mapper_seeds, mapper_cases = [], [], []
for case in cases_run:
    current = state[case['case_id']]
    try:
        mapper_messages.append(checked_messages(
            ISSUE_MAPPER_SYSTEM_PROMPT,
            build_issue_prompt(case['case_query'], current['laws']),
            MAX_NEW_TOKENS_ISSUE,
        ))
        mapper_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'issue_mapper'))
        mapper_cases.append(case)
    except Exception as exc:
        current['stage_errors'].append({'stage': 'issue_mapper_input', 'error': repr(exc)})
        current['issue_map'] = fallback_issue_map(case['case_query'], exc)

if mapper_cases:
    try:
        mapper_outputs = vllm_generate(
            mapper_messages, mapper_seeds, ISSUE_SCHEMA, MAX_NEW_TOKENS_ISSUE,
            enable_thinking=False,
        )
        if len(mapper_outputs) != len(mapper_cases):
            raise RuntimeError(f'Mapper returned {len(mapper_outputs)}/{len(mapper_cases)} outputs')
        for case, (text, usage) in zip(mapper_cases, mapper_outputs):
            current = state[case['case_id']]
            current['raw']['issue_mapper'] = text
            current['usage']['issue_mapper'] = usage
            try:
                current['issue_map'] = validate_issue_map(_json_object(text), case['case_query'])
            except Exception as exc:
                current['stage_errors'].append({'stage': 'issue_mapper_output', 'error': repr(exc)})
                current['issue_map'] = fallback_issue_map(case['case_query'], exc)
    except Exception as exc:
        for case in mapper_cases:
            current = state[case['case_id']]
            current['stage_errors'].append({'stage': 'issue_mapper_infrastructure', 'error': repr(exc)})
            current['issue_map'] = fallback_issue_map(case['case_query'], exc)

for case in cases_run:
    current = state[case['case_id']]
    if current['issue_map'] is None:
        current['issue_map'] = fallback_issue_map(case['case_query'], 'mapper produced no state')

# Stage 2: three independent compact Judge votes. A failed vote does not kill the case.
for vote_index in range(JUDGE_VOTES):
    vote_messages, vote_seeds, vote_cases = [], [], []
    for case in cases_run:
        current = state[case['case_id']]
        try:
            vote_messages.append(checked_messages(
                JUDGE_SYSTEM_PROMPT,
                build_judge_prompt(
                    case['case_query'], current['laws'], current['issue_map'], case['case_facts']
                ),
                MAX_NEW_TOKENS_JUDGE,
            ))
            vote_seeds.append(deterministic_stage_seed(
                EVAL_SEED, case['case_id'], f'legal_judge_vote_{vote_index}'
            ))
            vote_cases.append(case)
        except Exception as exc:
            current['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_input', 'error': repr(exc)
            })
    if not vote_cases:
        continue
    try:
        vote_outputs = vllm_generate(
            vote_messages, vote_seeds, JUDGE_SCHEMA, MAX_NEW_TOKENS_JUDGE
        )
        if len(vote_outputs) != len(vote_cases):
            raise RuntimeError(f'Judge vote {vote_index} returned {len(vote_outputs)}/{len(vote_cases)} outputs')
        for case, (text, usage) in zip(vote_cases, vote_outputs):
            current = state[case['case_id']]
            stage = f'legal_judge_vote_{vote_index}'
            current['raw'][stage] = text
            current['usage'][stage] = usage
            try:
                vote = validate_judge(
                    _json_object(text), current['issue_map'], current['laws']
                )
                vote['vote_index'] = vote_index
                current['votes'].append(vote)
            except Exception as exc:
                current['stage_errors'].append({'stage': stage, 'error': repr(exc)})
    except Exception as exc:
        for case in vote_cases:
            state[case['case_id']]['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_infrastructure', 'error': repr(exc)
            })

# Direct fallback only for cases with zero usable Judge votes.
direct_cases = [case for case in cases_run if not state[case['case_id']]['votes']]
if direct_cases:
    direct_messages, direct_seeds, runnable_direct = [], [], []
    for case in direct_cases:
        current = state[case['case_id']]
        try:
            direct_messages.append(checked_messages(
                DIRECT_SYSTEM_PROMPT,
                build_direct_prompt(case['case_query'], current['laws'], case['case_facts']),
                MAX_NEW_TOKENS_DIRECT,
            ))
            direct_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'direct_fallback'))
            runnable_direct.append(case)
        except Exception as exc:
            current['stage_errors'].append({'stage': 'direct_fallback_input', 'error': repr(exc)})
    if runnable_direct:
        try:
            direct_outputs = vllm_generate(
                direct_messages, direct_seeds, DIRECT_SCHEMA, MAX_NEW_TOKENS_DIRECT,
                enable_thinking=False,
            )
            if len(direct_outputs) != len(runnable_direct):
                raise RuntimeError(f'Direct fallback returned {len(direct_outputs)}/{len(runnable_direct)} outputs')
            for case, (text, usage) in zip(runnable_direct, direct_outputs):
                current = state[case['case_id']]
                current['raw']['direct_fallback'] = text
                current['usage']['direct_fallback'] = usage
                try:
                    current['direct'] = validate_direct(_json_object(text))
                except Exception as exc:
                    current['stage_errors'].append({'stage': 'direct_fallback', 'error': repr(exc)})
        except Exception as exc:
            for case in runnable_direct:
                state[case['case_id']]['stage_errors'].append({
                    'stage': 'direct_fallback_infrastructure', 'error': repr(exc)
                })

duration_all = round(time.time() - started_all, 3)
per_case_duration = round(duration_all / max(1, len(cases_run)), 3)
for case in cases_run:
    cid = case['case_id']
    current = state[cid]
    if current['votes']:
        verdict = aggregate_votes(current['votes'])
        prediction_source = (
            'judge_ensemble' if len(current['votes']) >= 2 else 'judge_single_vote'
        )
    elif current.get('direct'):
        verdict = {
            **current['direct'],
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'direct_fallback'
    else:
        verdict = {
            'prediction': HARD_FALLBACK_LABEL,
            'confidence': 0.0,
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'hard_fallback'

    # ==== DECIDER: nhan cuoi suy tu claim_outcomes, khong dung nhan model tu khai ====
    # TANG 1 (giu nguyen, nghien cuu cu): derived (tu claim_outcomes CUA 1 VOTE DAI DIEN,
    # chon boi aggregate_votes theo confidence) thang holistic. Do tren lan chay truoc
    # (mapper hoat dong, 2.3 claim/case, 33/50 case >=2 claim):
    #   holistic  strict .540 / macroF1 .508  |  derived  strict .640 / macroF1 .650
    #   nested CV 5-fold x200: holistic .445  |  derived .557  (base Qwen3.5-9B .405)
    #   hieu so cap derived-holistic: CI95 [.018, .266], P(derived > holistic) = .991
    # Quet 216 cau hinh nguong: 216/216 deu thang holistic (thap nhat .524) => ket luan
    # khong phu thuoc viec chon nguong. KHONG chinh nguong .85/.55/.30: chinh lam TE di
    # (chon nguong trong fold tut ve .526).
    holistic_label = verdict['prediction']
    derived_final = (
        derive_label(current['issue_map'], verdict['claim_outcomes'])
        if verdict['claim_outcomes'] else None
    )
    base_prediction = derived_final or holistic_label

    # TANG 2 (MOI, 2026-07-24): TANG 1 chi self-consistency o cap HOLISTIC (aggregate_votes
    # dung nhieu vote de CHON 1 vote dai dien, roi derive_label CHI tren claim_outcomes cua
    # RIENG vote do -- van la 1 phieu duy nhat quyet dinh derived_final). validate_judge da
    # tinh SAN derive_label cho TUNG vote rieng le (vote['derived_label']) nhung TANG 1 khong
    # dung het -- day la cho co the ap self-consistency THEM 1 lan nua, o dung cap derived.
    # Quy tac: lay MAJORITY cua ca N derived_label (N=JUDGE_VOTES); CHI ghi de base_prediction
    # khi co da so TUYET DOI ro rang (mot nhan chiem >=2 phieu VA khong hoa voi nhan khac).
    # Hoa phieu -> GIU base_prediction (khong tie-break vo can cu bang confidence).
    # Do THAT tren public 50 case (fullchunks/pado, qwen3.5-9b, phien 2026-07-24):
    #   base_prediction (= TANG 1, nhu file dang nop)   strict .580 / bin .740
    #   majority(derived_label_votes) THAN TRONG         strict .620 / bin .780  <-- dung
    # Dung HET logic da dung hau xu ly file nop private (build_submission_majorityvote.py) --
    # sua o day de lan chay GPU tiep theo ra THANG nhan nay tu dau, khong can vet lai.
    derived_label_votes = [vote.get('derived_label') for vote in current['votes']]
    _derived_counts = {label: 0 for label in LABELS}
    for _lbl in derived_label_votes:
        if _lbl in LABELS:
            _derived_counts[_lbl] += 1
    _derived_max = max(_derived_counts.values())
    _derived_candidates = [label for label in LABELS if _derived_counts[label] == _derived_max]
    if _derived_max >= 2 and len(_derived_candidates) == 1:
        final_prediction = _derived_candidates[0]
        decider = 'majority_derived_votes'
    else:
        final_prediction = base_prediction
        decider = 'derived_from_claim_outcomes' if derived_final else 'holistic_fallback'

    result = {
        'case_id': cid,
        'case_query': case['case_query'],
        'eval_seed': EVAL_SEED,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'pipeline_stages': {
            'issue_mapper': current['issue_map'],
            'judge_votes': current['votes'],
        },
        'retrieved_laws': current['laws'],
        'raw_responses': current['raw'],
        'usage': aggregate_stage_usage(current['usage']),
        'duration_seconds': per_case_duration,
        'cache_key': cache_keys[cid],
        'prediction': final_prediction,
        'prediction_source': prediction_source,
        'decider': decider,
        'holistic_label': holistic_label,
        'derived_label_final': derived_final,
        'confidence': verdict['confidence'],
        'reasoning': (
            f"source={prediction_source}; votes={verdict['vote_counts']}; "
            f"valid_votes={verdict['n_valid_votes']}"
        ),
        'applied_laws': verdict['applied_laws'],
        'claim_outcomes': verdict['claim_outcomes'],
        'vote_counts': verdict['vote_counts'],
        'n_valid_votes': verdict['n_valid_votes'],
        # Giu de cham doi chung offline (holistic_label vs prediction).
        'derived_label_votes': derived_label_votes,
        'mapper_fallback': bool(current['issue_map'].get('mapper_fallback')),
        'stage_errors': current['stage_errors'],
        'failed_stage': None,
        'error_kind': None,
        'error': None,
    }
    saved[cid] = result

    # Checkpoint từng case: nếu notebook bị ngắt sau đó, rerun không phải sinh lại case đã hoàn tất.
    atomic_write_json(result_path, saved)

# ==== TANG 1 (majority-at-claim): quyet dinh nhan cuoi tu SU DONG THUAN cac phieu ====
# Hau ky thuan tuy - chi doc judge_votes DA LUU, KHONG goi model. Nen chay lai voi cache
# khop (0 case trong cases_run) van cap nhat duoc nhan ma khong ton GPU. Ap cho MOI case
# trong saved, ke ca case nap tu checkpoint. prediction = derive(consensus) thay vi
# derive(phieu dai dien). Giu holistic_label + phieu dai dien de cham doi chung offline.
for _cid, _res in saved.items():
    _stages = _res.get('pipeline_stages') or {}
    _votes = _stages.get('judge_votes') or []
    _imap = _stages.get('issue_mapper') or {'claims': []}
    if not (_votes and _imap.get('claims')):
        continue
    _consensus = consensus_claim_outcomes(_votes, _imap)
    _label = derive_label(_imap, _consensus)
    if not _label:
        continue
    _res.setdefault('holistic_label', _res.get('prediction'))
    _res.setdefault('representative_claim_outcomes', _res.get('claim_outcomes'))
    _res['consensus_claim_outcomes'] = _consensus
    _res['claim_outcomes'] = _consensus
    _res['derived_label_final'] = _label
    _res['prediction'] = _label
    _res['decider'] = 'majority_at_claim_consensus'
expected_ids = {case['case_id'] for case in inference_cases}
missing_ids = sorted(expected_ids - set(saved))
invalid_ids = sorted(
    cid for cid in expected_ids if (saved.get(cid) or {}).get('prediction') not in LABELS
)
if missing_ids or invalid_ids:
    raise RuntimeError(f'Coverage invariant failed: missing={missing_ids}, invalid={invalid_ids}')
atomic_write_json(result_path, saved)
all_model_results = {MODEL: {EVAL_SEED: saved}}
mapper_fallbacks = sum(1 for v in saved.values() if v.get('mapper_fallback'))
claim_counts = {}
for value in saved.values():
    n = len((value.get('pipeline_stages') or {}).get('issue_mapper', {}).get('claims') or [])
    claim_counts[n] = claim_counts.get(n, 0) + 1
print(f'Issue mapper: fallback={mapper_fallbacks}/{len(saved)} | claims/case={dict(sorted(claim_counts.items()))}')
source_counts = {}
for value in saved.values():
    source = value.get('prediction_source', 'unknown')
    source_counts[source] = source_counts.get(source, 0) + 1
print(
    f'Completed v12: {sum(v.get("prediction") in LABELS for v in saved.values())}/'
    f'{len(inference_cases)} valid | sources={source_counts} | '
    f'{duration_all}s for {len(cases_run)} cases (~{per_case_duration}s/case).'
)


Benchmark manifest: outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge/benchmark_manifest_qwen3-5-9b.json
=== v12 compact ensemble | seed=2026 | run 50/50 cases ===


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|            | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=462) WARNING 08-10 02:59:21 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|█| 50/50 [00:46<00:00,  1.08it/s, est. speed input: 8638.39 toks/s, output: 102.95 toks


...

Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts: 100%|█| 50/50 [08:37<00:00, 10.35s/it, est. speed input: 1007.77 toks/s, output: 814.18 toks


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts: 100%|█| 50/50 [08:34<00:00, 10.29s/it, est. speed input: 1013.49 toks/s, output: 828.99 toks


Issue mapper: fallback=2/50 | claims/case={1: 17, 2: 16, 3: 11, 4: 4, 5: 2}
Completed v12: 50/50 valid | sources={'judge_ensemble': 50} | 1612.574s for 50 cases (~32.251s/case).


In [10]:
def evaluate_run(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        prediction = item.get('prediction')
        is_valid_output = prediction in LABELS
        usage = item.get('usage') or {}
        rows.append({
            'model': model,
            'seed': eval_seed,
            'case_id': cid,
            'gold': gold_by_case[cid],
            'prediction': prediction,
            'scored_prediction': prediction if is_valid_output else INVALID_OUTPUT_LABEL,
            'is_valid_output': is_valid_output,
            'error_kind': item.get('error_kind'),
            'is_infrastructure_failure': item.get('error_kind') == 'infrastructure',
            'confidence': item.get('confidence'),
            'prediction_source': item.get('prediction_source'),
            'mapper_fallback': bool(item.get('mapper_fallback', False)),
            'used_model_fallback': item.get('prediction_source') in {'direct_fallback', 'hard_fallback'},
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'duration_seconds': item.get('duration_seconds'),
            'error': item.get('error'),
        })
    frame = pd.DataFrame(rows)
    valid = frame[frame['is_valid_output']].copy()
    n_total, n_valid = len(frame), len(valid)
    n_failed = n_total - n_valid
    n_infrastructure_failed = int(frame['is_infrastructure_failure'].sum())
    if n_failed:
        failed_ids = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
        print(
            f'Cảnh báo {model} seed={eval_seed}: {n_failed}/{n_total} output không hợp lệ '
            f'được tính sai. Case: {failed_ids}'
        )

    scored_prediction = frame['scored_prediction']
    strict_correct = int((frame['gold'] == scored_prediction).sum())
    strict_accuracy = strict_correct / n_total if n_total else 0.0
    valid_accuracy = accuracy_score(valid['gold'], valid['prediction']) if n_valid else 0.0
    report = classification_report(
        frame['gold'], scored_prediction, labels=LABELS,
        output_dict=True, zero_division=0,
    ) if n_total else {}
    cm_all = confusion_matrix(
        frame['gold'], scored_prediction, labels=LABELS + [INVALID_OUTPUT_LABEL]
    ) if n_total else np.zeros((len(LABELS) + 1, len(LABELS) + 1), dtype=int)
    cm = cm_all[:len(LABELS), :]
    summary = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model,
        'seed': eval_seed,
        'n_total': n_total,
        'n_success': n_valid,
        'n_failed': n_failed,
        'n_infrastructure_failed': n_infrastructure_failed,
        'invalid_output_rate': n_failed / n_total if n_total else 0.0,
        'coverage': n_valid / n_total if n_total else 0.0,
        'benchmark_valid': (
            n_total == len(inference_cases)
            and n_infrastructure_failed == 0
            and not bool((frame['prediction_source'] == 'hard_fallback').any())
        ),
        'all_outputs_valid': n_valid == n_total,
        'metric_scope': 'all_50_invalid_outputs_count_as_wrong',
        'strict_accuracy_all_50': strict_accuracy,
        'accuracy_successful_only': valid_accuracy,
        'macro_precision': report.get('macro avg', {}).get('precision', 0.0),
        'macro_recall': report.get('macro avg', {}).get('recall', 0.0),
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0.0),
        'weighted_f1': report.get('weighted avg', {}).get('f1-score', 0.0),
        'avg_input_tokens': frame['input_tokens'].mean(),
        'avg_output_tokens': frame['output_tokens'].mean(),
        'avg_duration_seconds': frame['duration_seconds'].mean(),
        'n_hit_max_new_tokens': int(frame['hit_max_new_tokens'].sum()),
        'n_mapper_fallback': int(frame['mapper_fallback'].sum()),
        'n_model_fallback': int(frame['used_model_fallback'].sum()),
        'n_hard_fallback': int((frame['prediction_source'] == 'hard_fallback').sum()),
        'prediction_sources': frame['prediction_source'].value_counts(dropna=False).to_dict(),
    }
    per_label = pd.DataFrame([
        {
            'model': model,
            'seed': eval_seed,
            'label': label,
            'precision': report.get(label, {}).get('precision', 0.0),
            'recall': report.get(label, {}).get('recall', 0.0),
            'f1': report.get(label, {}).get('f1-score', 0.0),
            'support': int(report.get(label, {}).get('support', 0)),
        } for label in LABELS
    ])
    cm_frame = pd.DataFrame(
        cm,
        index=[f'gold_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS + [INVALID_OUTPUT_LABEL]],
    )
    return summary, per_label, cm_frame, frame

summaries = []
evaluation_artifacts = {}
for model, seed_results in all_model_results.items():
    evaluation_artifacts[model] = {}
    for eval_seed, results in seed_results.items():
        summary, per_label, cm_frame, case_frame = evaluate_run(model, eval_seed, results)
        summaries.append(summary)
        evaluation_artifacts[model][eval_seed] = {
            'per_label': per_label,
            'confusion_matrix': cm_frame,
            'cases': case_frame,
        }
        print(f'\n=== {model} | seed={eval_seed} ===')
        display(pd.DataFrame([summary]))
        display(cm_frame)

run_metrics = pd.DataFrame(summaries).sort_values(['model', 'seed']).reset_index(drop=True)
aggregate_metrics = run_metrics.groupby('model', as_index=False).agg(
    n_seeds=('seed', 'nunique'),
    accuracy_mean=('strict_accuracy_all_50', 'mean'),
    accuracy_std=('strict_accuracy_all_50', 'std'),
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    invalid_rate_mean=('invalid_output_rate', 'mean'),
    invalid_rate_std=('invalid_output_rate', 'std'),
    avg_output_tokens=('avg_output_tokens', 'mean'),
    avg_duration_seconds=('avg_duration_seconds', 'mean'),
)
aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']] = (
    aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']].fillna(0.0)
)
leaderboard = aggregate_metrics.sort_values(
    ['accuracy_mean', 'macro_f1_mean'], ascending=False
).reset_index(drop=True)

majority_label = pd.Series(list(gold_by_case.values())).value_counts().idxmax()
majority_accuracy = pd.Series(list(gold_by_case.values())).value_counts().max() / len(gold_by_case)
print(f'Majority baseline: {majority_label} | accuracy={majority_accuracy:.4f}')
print('Per-seed metrics:')
display(run_metrics)
print('Aggregate mean ± std across seeds:')
display(leaderboard)

for model, seed_artifacts in evaluation_artifacts.items():
    slug = slugify(model)
    model_runs = run_metrics[run_metrics['model'] == model]
    model_summary = leaderboard[leaderboard['model'] == model]
    model_runs.to_csv(
        OUTPUT_DIR / f'model_metrics_by_seed_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    model_summary.to_csv(
        OUTPUT_DIR / f'model_metrics_summary_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    for eval_seed, artifacts in seed_artifacts.items():
        suffix = f'{slug}_seed-{eval_seed}'
        artifacts['per_label'].to_csv(
            OUTPUT_DIR / f'metrics_per_label_{suffix}.csv', index=False, encoding='utf-8-sig'
        )
        artifacts['confusion_matrix'].to_csv(
            OUTPUT_DIR / f'confusion_matrix_{suffix}.csv', encoding='utf-8-sig'
        )
        artifacts['cases'].to_csv(
            OUTPUT_DIR / f'case_predictions_{suffix}.csv', index=False, encoding='utf-8-sig'
        )

# ---- Paired comparison với predictions của base trên đúng 50 case ----
base_prediction_name = f'predictions_qwen3-4b_seed-{EVAL_SEED}.json'
base_candidates = []
if os.getenv('ALQAC_BASE_PREDICTIONS_PATH'):
    base_candidates.append(Path(os.environ['ALQAC_BASE_PREDICTIONS_PATH']))
base_candidates += [
    DATA_PATH.parent / base_prediction_name,
    PROJECT_ROOT / base_prediction_name,
    PROJECT_ROOT / 'bge' / 'base' / base_prediction_name,
    Path.cwd() / 'bge' / 'base' / base_prediction_name,
    Path('/root/bge/base') / base_prediction_name,
    Path.cwd() / base_prediction_name,
]
base_prediction_path = next((path.resolve() for path in base_candidates if path.is_file()), None)
if base_prediction_path is None:
    print('Không tìm thấy base predictions; bỏ qua paired comparison. Candidates:', base_candidates)
else:
    base_results = json.loads(base_prediction_path.read_text(encoding='utf-8'))
    pado_results = all_model_results[MODEL][EVAL_SEED]
    paired_rows = []
    for case in inference_cases:
        cid = case['case_id']
        base_prediction = (base_results.get(cid) or {}).get('prediction')
        pado_prediction = (pado_results.get(cid) or {}).get('prediction')
        gold = gold_by_case[cid]
        paired_rows.append({
            'case_id': cid, 'gold': gold,
            'base_prediction': base_prediction, 'pado_prediction': pado_prediction,
            'base_correct': base_prediction == gold,
            'pado_correct': pado_prediction == gold,
            'changed': base_prediction != pado_prediction,
        })
    paired = pd.DataFrame(paired_rows)
    paired_summary = {
        'base_path': str(base_prediction_path),
        'n_cases': len(paired),
        'base_accuracy': float(paired['base_correct'].mean()),
        'pado_accuracy': float(paired['pado_correct'].mean()),
        'accuracy_delta': float(paired['pado_correct'].mean() - paired['base_correct'].mean()),
        'n_changed': int(paired['changed'].sum()),
        'n_improved': int((~paired['base_correct'] & paired['pado_correct']).sum()),
        'n_regressed': int((paired['base_correct'] & ~paired['pado_correct']).sum()),
    }
    transition = pd.crosstab(
        paired['base_prediction'].fillna(INVALID_OUTPUT_LABEL),
        paired['pado_prediction'].fillna(INVALID_OUTPUT_LABEL),
        rownames=['base'], colnames=['pado'], dropna=False,
    )
    paired.to_csv(OUTPUT_DIR / 'paired_cases_vs_base.csv', index=False, encoding='utf-8-sig')
    transition.to_csv(OUTPUT_DIR / 'prediction_transition_vs_base.csv', encoding='utf-8-sig')
    atomic_write_json(OUTPUT_DIR / 'paired_summary_vs_base.json', paired_summary)
    print('Paired comparison vs base:')
    display(pd.DataFrame([paired_summary]))
    display(transition)



=== Qwen3.5-9B | seed=2026 ===


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,n_infrastructure_failed,invalid_output_rate,coverage,...,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens,n_mapper_fallback,n_model_fallback,n_hard_fallback,prediction_sources
0,v12_legal_pado_divisible_types_balanced_judge,legal_pado_thinking_divisible_mapper_3vote_der...,Qwen3.5-9B,2026,50,50,0,0,0.0,1.0,...,0.59949,0.575174,39291.86,25343.82,32.251,0,2,0,0,{'judge_ensemble': 50}


,pred_A_WIN,pred_B_WIN,pred_PARTIAL_A_WIN,pred_PARTIAL_B_WIN,pred___INVALID_OUTPUT__
gold_A_WIN,9,1,5,1,0
gold_B_WIN,2,7,1,0,0
gold_PARTIAL_A_WIN,4,4,9,2,0
gold_PARTIAL_B_WIN,0,0,1,4,0


Majority baseline: PARTIAL_A_WIN | accuracy=0.3800
Per-seed metrics:


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,n_infrastructure_failed,invalid_output_rate,coverage,...,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens,n_mapper_fallback,n_model_fallback,n_hard_fallback,prediction_sources
0,v12_legal_pado_divisible_types_balanced_judge,legal_pado_thinking_divisible_mapper_3vote_der...,Qwen3.5-9B,2026,50,50,0,0,0.0,1.0,...,0.59949,0.575174,39291.86,25343.82,32.251,0,2,0,0,{'judge_ensemble': 50}


Aggregate mean ± std across seeds:


,model,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,invalid_rate_std,avg_output_tokens,avg_duration_seconds
0,Qwen3.5-9B,1,0.58,0.0,0.59949,0.0,0.0,0.0,25343.82,32.251


Không tìm thấy base predictions; bỏ qua paired comparison. Candidates: [PosixPath('/root/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/predictions_qwen3-4b_seed-2026.json')]


In [11]:
run_metrics[['strict_accuracy_all_50', 'macro_precision', 'macro_recall', 'macro_f1']]


,strict_accuracy_all_50,macro_precision,macro_recall,macro_f1
0,0.58,0.579315,0.634046,0.59949


In [12]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                # Khop cau truc voi ban da NOP cho PRIVATE (2026-07-24) de do duoc tren
                # public (co gold that) truoc khi tin tuong dung lai cho private.
                'case_evidence': evidence_ids_by_case.get(case['case_id'], []),
                # law_evidence = TOAN BO dieu luat retrieval cua case, KHONG chi applied_laws
                # (judge gan nhu khong bao gio dien legal_basis_ranks -> applied_laws thua thiet).
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in retrieval_cache.get(case['case_id'], [])
                ],
            })
        expected_ids = {case['case_id'] for case in inference_cases}
        submission_ids = [row['case_id'] for row in submission]
        if len(submission) != len(inference_cases) or set(submission_ids) != expected_ids:
            raise RuntimeError(
                f'Submission coverage failed: rows={len(submission)}, unique={len(set(submission_ids))}'
            )
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| evaluated:', len(inference_cases), '/ 50',
            '| invalid counted wrong:', n_failed,
            '| valid submission rows:', len(submission), '/ 50',
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


Qwen3.5-9B | seed: 2026 | evaluated: 50 / 50 | invalid counted wrong: 0 | valid submission rows: 50 / 50 | outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge/submission_qwen3-5-9b_seed-2026.json
Outputs: /root/outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge


In [13]:
# Dong Modal app: giai phong container GPU remote sau khi chay xong toan bo pipeline.
close_modal()
print('Da dong Modal app.')


(EngineCore pid=462) INFO 08-10 03:28:04 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=462) INFO 08-10 03:28:04 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=462) INFO 08-10 03:28:04 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=462) INFO 08-10 03:28:04 [core.py:1227] [shutdown] EngineCore: exiting busy loop


(EngineCore pid=462) Process EngineCore:
(EngineCore pid=462) Traceback (most recent call last):
(EngineCore pid=462)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1224, in run_engine_core
(EngineCore pid=462)     engine_core.run_busy_loop()
(EngineCore pid=462)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1267, in run_busy_loop
(EngineCore pid=462)     raise SystemExit
(EngineCore pid=462) SystemExit
(EngineCore pid=462) 
(EngineCore pid=462) During handling of the above exception, another exception occurred:
(EngineCore pid=462) 
(EngineCore pid=462) Traceback (most recent call last):
(EngineCore pid=462)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=462)     self.run()
(EngineCore pid=462)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=462)     self._target(*self._args, **self._kwargs)
(EngineCore pid=462)   File "

WARNING 08-10 03:28:09 [utils.py:626] [shutdown] Process manager: force killing remaining processes count=1
Da giai phong GPU local.
Da dong Modal app.
